# Explicabilidad y simulador de riesgo por vano

Hermano de `04_uiti_vano_trayectorias_vano.ipynb`: reutiliza su clasificacion KMeans por
vano x ventana -- que nunca se reajusta aqui -- y agrega encima un simulador de "que
pasaria si" a nivel de vano.

**Un solo modelo y una sola unidad.** Las tres salidas del boton "Simular" -- el mapa
**Criticidad Simulada**, el **Grafo reconstruido** y **Importancia Variables** -- vienen
del MIL entrenado en `05_mil_vano_ventana`, que puntua **bolsas**: una bolsa es una celda
(vano, ventana) y sus instancias son los eventos de ese vano en esa ventana. La clase sale
de `asignar_clase(n_obs observado, u-hat predicho)` sobre la geometria KMeans de 04, la
misma del mapa base, de modo que los dos mapas comparten paleta por construccion y no por
convencion. `n_obs` nunca se simula: es un eje del espacio que define la clase.

**Que mide cada mapa.** **Criticidad Original** (fila 1) es el grupo historico que 04 ya
calculo sobre eventos observados. **Criticidad Simulada** (fila 2) es lo que el modelo
predice al aplicar las variables del simulador. Son dos mediciones distintas y nunca
comparten leyenda ni titulo.

**Requiere dos artefactos de `05_mil_vano_ventana.ipynb`**: `data/models/mil_vano_ventana_v1.pt`
y `data/derived/bolsas_mil_full.joblib`. Los dos viven bajo `data/`, que git ignora; si
faltan, la celda del modelo falla de inmediato nombrando el cuaderno que los produce.

**La interfaz es este cuaderno.** Todo se controla y se ve en la figura de 4x3 paneles y
sus 46 trazas, con los controles de `ipywidgets` encima. Al ejecutar no se escribe ningun
archivo ni se abre ningun navegador. Quien quiera ademas un HTML autocontenido para
compartir tiene `EXPORTAR_PANEL_WEB`, descrito al final.


In [ ]:
import asyncio
import os
import sys
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError('Este cuaderno requiere ipywidgets para la interfaz interactiva.') from exc
from IPython.display import display

# Sube desde el cwd hasta la raiz del repo (marcada por la carpeta src/), igual que 09.
# Se agregan ROOT y ROOT/src -- no solo src/ -- porque ventanas_015.py importa
# `scripts.extract_geometrias_014` (paquete de nivel de repo, igual que en notebook 10).
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for _path_a_agregar in (ROOT, ROOT / 'src'):
    if str(_path_a_agregar) not in sys.path:
        sys.path.insert(0, str(_path_a_agregar))

# Un kernel que ya importo estos paquetes se queda con la version VIEJA en `sys.modules`:
# "Run All" sin reiniciar NO vuelve a leer el disco. Un rename en src/ -- por ejemplo
# `SelectorVanos._caja` -> `.caja` -- estalla entonces como AttributeError diez celdas mas
# abajo, con el codigo del disco ya correcto. Se purgan ANTES de importarlos, asi el
# cuaderno corre SIEMPRE contra la fuente actual, con o sin reinicio de kernel. Va aqui y no
# como `importlib.reload`: reload no rehace los objetos ya construidos con la clase vieja,
# y este cuaderno los reconstruye todos de esta celda para abajo.
for _modulo in [m for m in list(sys.modules)
                if m.split('.')[0] in ('chec_impacto', 'chec_local_interpreter', 'scripts')]:
    del sys.modules[_modulo]

from chec_impacto.data import procesar_dataset_completo
from chec_impacto.models.criticality_assignment import (
    CLAVE_ESPACIO_CANONICO,
    GEOMETRIAS_SHA1_ESPERADO,
    cargar_geometria_014,
    verificar_sha1_geometrias,
)
from chec_impacto.data.bags import cargar_bolsas
from chec_impacto.models.mil_persistencia import cargar_modelo_mil
from chec_impacto.training import resolve_training_device
from chec_local_interpreter.mil_simulador_015 import (
    gates_de_bolsas,
    grafo_de_gates,
    seleccionar_bolsas,
    simular_bolsas,
    trazas_grafo,
)
from chec_local_interpreter.vano_controls import build_knobs, expand_knob_overrides
from chec_local_interpreter.vano_widgets import (
    construir_selector_casillas,
    construir_selector_vanos,
)
from chec_local_interpreter.ventanas_015 import (
    capas_mapa_historico,
    cargar_clases_desde_014,
    centro_y_zoom,
    construir_hist_class_cache,
    construir_mask_cache,
    construir_tabla_vano_ventana,
    construir_ventanas,
    fid_de_punto,
    frontera_kmeans,
    nube_fondo,
    nube_seleccion,
    reparto_por_clase,
    series_temporal_vanos,
)
from scripts.extract_geometrias_014 import (
    DEFAULT_NOTEBOOK_PATH,
    DEFAULT_OUTPUT_PATH,
    extraer_geometrias_014,
)

# Sonda del contrato que rompio el cuaderno dos veces. Si el kernel siguiera sirviendo una
# version vieja de vano_widgets, falla AQUI -- primera celda, mensaje que dice que hacer --
# en vez de a los 10 minutos de procesamiento, en la celda del panel.
_sonda = construir_selector_vanos(['0'])
assert hasattr(_sonda, 'caja'), (
    'vano_widgets viejo en memoria: el selector de casillas sin `.caja`. '
    'Reinicia el kernel. '
    f'(modulo cargado desde {sys.modules["chec_local_interpreter.vano_widgets"].__file__})'
)
del _sonda

In [ ]:
# Ventana climatica igual que 03_mgcecdl_training / 09_simulador: cambiarla generaria un
# set de features distinto al que el modelo cargado en la celda SEAM espera.
VENTANA_CLIMATICA_HORAS = 12
CLAVE_ESPACIO = CLAVE_ESPACIO_CANONICO  # '2' -- espacio canonico fijado en criticality_assignment.py
DEVICE = resolve_training_device('auto')

# Misma paleta que 01.4: los grupos historicos de este cuaderno SON los de 01.4, nunca se
# reajustan, asi que el color tiene que significar lo mismo en los dos cuadernos.
NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
# UN solo codigo de ausencia en los DOS mapas: negro, el `COLOR_SIN_EVENTO` de 01.4.
# El vano sin eventos en la ventana no tiene clase, y la ausencia no es la clase mas baja;
# que el base lo pintara gris y el simulado negro obligaba a recordar dos codigos para la
# misma cosa. Los cuatro colores KMeans significan lo mismo en los dos mapas.
COLOR_SIN_EVENTO = 'rgb(0,0,0)'
COLOR_MARCADO = '#0072b2'
# Equipos: mismos colores que 01.4, por el mismo motivo que la paleta de grupos --
# un naranja tiene que seguir siendo un transformador al pasar de un cuaderno a otro.
COLOR_TRAFO = '#f59e0b'
COLOR_SWITCH = '#7c3aed'
ANCHO_MAPA = 3.0
ANCHO_MAPA_MARCADO = round(ANCHO_MAPA * 1.4, 2)

# Fila 1, paridad 01.4: un vano MARCADO se dibuja con el color de SU clase, sobre un halo
# blanco que lo despega del fondo (01.4: `width=ANCHO_MAPA_RESALTE * 2.6, color='white'`).
# Un color plano de "seleccionado" encima de la clase congela lo que se ve: la ventana
# cambia la clase por debajo y el vano marcado sigue igual en pantalla. COLOR_MARCADO
# queda solo para la fila 2, donde la clase la pone el modelo y no el KMeans.
COLOR_HALO = 'white'
ANCHO_HALO = round(ANCHO_MAPA_MARCADO * 2.6, 2)
OPACIDAD_NUBE = 0.45               # 01.4, para que la nube de fondo no tape el resaltado
OPACIDAD_FRONTERA = 0.28           # 01.4: el contorno es fondo, no dato
# Paleta de 01.4 para las series por vano: apta para daltonismo y distinta de la escala de
# grupos, porque aqui el color identifica AL VANO, no a su clase.
COLORES_VANOS = ['#0072b2', '#009e73', '#cc79a7', '#56b4e9', '#e69f00', '#8c564b']
N_CUPOS_EVOLUCION = len(COLORES_VANOS)
# Paleta de los MODOS de variable en el grafo. Deliberadamente fuera de la familia de los
# grupos KMeans (rojos/naranjas) y de los equipos: un rojo en el mapa y un rojo en el grafo
# significarian cosas sin ninguna relacion. Se recorre en el orden de las modalidades del
# artefacto.
PALETA_MODALIDADES = ['#0d9488', '#be185d']   # verde azulado y rosa oscuro

# --- Panel web: OPCIONAL, apagado por defecto ---------------------------------------
# EL CUADERNO ES LA INTERFAZ. Circuito, ventana, seleccion de vanos, variables del
# simulador y el boton "Simular" se controlan y se ven en la app de widgets de mas abajo,
# tanto en local como en Databricks. Al ejecutar NO se escribe ningun archivo y no se abre
# ningun navegador.
#
# Poniendo esto en True, la ultima celda ADEMAS arma un HTML autocontenido (~12 MB, con
# `plotly.js` y el simulador adentro) para abrir a pantalla completa o compartir con quien
# no tenga el entorno. Es un entregable aparte, no parte de correr el cuaderno.
EXPORTAR_PANEL_WEB = False

# `DATABRICKS_RUNTIME_VERSION` la define SIEMPRE el runtime y no existe fuera de el, asi
# que sirve igual en un cuaderno interactivo que en un job. No se mira `dbutils` ni
# `spark`: son nombres inyectados en el kernel del cuaderno y no existen en un `%run` ni
# en un modulo importado, con lo que la deteccion fallaria justo donde importa.
EN_DATABRICKS = bool(os.environ.get('DATABRICKS_RUNTIME_VERSION'))
# Solo tiene efecto con EXPORTAR_PANEL_WEB = True. En Databricks nunca abre nada: el
# driver es un contenedor sin escritorio ni navegador, y `webbrowser.open` ahi no le
# muestra el panel a nadie -- devuelve False, o intenta lanzar un `xdg-open` que no existe.
ABRIR_EN_NAVEGADOR = not EN_DATABRICKS
# Guion horizontal negro en cada extremo de CADA vano, tenga o no eventos, igual que el
# mapa de 01: grados de longitud a cada lado del extremo (~14 m a esta latitud). Marca
# donde empieza y donde termina un vano, que es lo unico que distingue dos vanos vecinos
# dibujados con el mismo color.
MARCA_VANO = 0.00013
# Densificacion del hover, SOLO en el panel web (se calcula en el navegador). El hover de
# una traza de lineas en Scattermap se resuelve contra los VERTICES y no contra la linea,
# y los tramos de MVLINSEC traen exactamente 2 vertices: el centro de un vano largo queda
# a mas de `hoverdistance` de los dos extremos y no muestra ninguna etiqueta. Medido, hacer
# esto del lado de Python llevaria el peor circuito de 4.131 a 22.371 puntos y ~2,8 MB de
# etiquetas por capa -- por encima del `iopub_data_rate_limit` del comm del widget.
PASO_VERTICE = 0.00022      # grados ~= 25 m a esta latitud
MAX_CORTES_TRAMO = 600      # techo por tramo, para el vano de 12 km

# --- Que circuitos pueden SIMULARSE en el navegador ----------------------------------
# El boton "Simular" del HTML corre el mismo modelo MIL que el cuaderno, pero en JS. Los
# PESOS son baratos (89.658 parametros vivos = 350 KB) y viajan siempre; lo que pesa es la
# matriz de instancias, que para los 208 circuitos son 88 MB en float32 -- imposible en un
# archivo unico. Por circuito baja a 0,26 MB (mediana), 1,24 (p95) y 5,32 (el peor), asi
# que se embarcan solo los circuitos elegidos. `None` = el circuito activo al exportar.
# Poner una lista mas larga es valido y la celda del panel imprime cuanto sumo.
CIRCUITOS_SIMULABLES = None


In [ ]:
# --- Reutilizacion de la geometria KMeans de 01.4 (design section F) -------
# Falla RAPIDO aqui, antes de procesar el dataset completo (celda siguiente): si 01.4 fue
# editado y sus centroides se movieron, no tiene sentido esperar el procesamiento pesado
# para enterarse. `cargar_clases_desde_014` (celda 7, via hist_class_cache) repite esta
# misma verificacion por cada ventana consultada -- barata, y evita que una geometria
# cacheada quede sin recomprobar dentro de la misma sesion.
GEOMETRIAS_PATH = DEFAULT_OUTPUT_PATH
if not GEOMETRIAS_PATH.exists():
    extraer_geometrias_014(DEFAULT_NOTEBOOK_PATH, GEOMETRIAS_PATH)
_sha1_real, _coincide = verificar_sha1_geometrias(GEOMETRIAS_PATH, esperado=GEOMETRIAS_SHA1_ESPERADO)
assert _coincide, (
    f'La geometria KMeans extraida de 01.4 no coincide con la esperada '
    f'(esperado={GEOMETRIAS_SHA1_ESPERADO}, real={_sha1_real}). 01.4 fue modificado; '
    f'01.5 depende de esa geometria.'
)
# La geometria en si (no solo su sha1): la nube KMeans de la fila 3 tiene que dibujarse en
# el MISMO espacio en que se asignan las clases -- el canonico '2' es (log_x=False,
# log_y=True). Leerlo de la geometria y no fijarlo a mano evita que un cambio de espacio
# deje la nube en ejes que ya no corresponden a las fronteras.
GEOMETRIA_014 = cargar_geometria_014(GEOMETRIAS_PATH, CLAVE_ESPACIO)
print(f'Geometria 01.4 verificada -- sha1 coincide ({_sha1_real[:12]}...) | '
      f'espacio {CLAVE_ESPACIO}: log_x={GEOMETRIA_014.logs[0]}, log_y={GEOMETRIA_014.logs[1]}')

In [ ]:
DATA_PATH = ROOT / 'data' / 'Indicadores_vano_v3.csv'
VARIABLES_SELECCION_PATH = ROOT / 'data' / 'Variables_seleccion.xlsx'
MODEL_DIR = ROOT / 'data' / 'models'

# Mismo preprocesamiento real usado en entrenamiento (03_mgcecdl_training / 09_simulador):
# sin muestreo ni filtro de UITI, para que context_df quede alineado FILA A FILA con X --
# la clave que permite reusar la MISMA mascara (circuito, ventana) para el mapa historico
# (sin modelo) y, en un PR futuro, para las predicciones del modelo sobre esas mismas filas.
datos = procesar_dataset_completo(
    path_clima=DATA_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

feature_names = list(datos['features'])
X_raw_model = np.asarray(datos['X'], dtype=np.float32)
Xdf = datos['Xdata'].copy().reset_index(drop=True)
context_df = datos['df_original_copy'].copy().reset_index(drop=True)
label_encoders = datos.get('label_encoders', {})
max_values_imputed = datos.get('max_values_imputed', {})

# Ya no se construye el escalador min-max de MGCECDL. El simulador y la importancia de
# variables corren sobre el modelo MIL del cuaderno 05, cuya matriz de instancias es RAW,
# asi que `preparar_splits_estratificados` + `escalar_features_minmax_mgcecdl` -- de lo
# mas caro de esta celda -- no alimentaban ya a nadie.
assert len(context_df) == len(X_raw_model), (
    'context_df y X_raw_model deben quedar alineados fila a fila'
)
print(f'{len(context_df):,} filas | {len(feature_names)} features')

In [ ]:
# --- UN solo modelo: el MIL por bolsas del cuaderno 05 (cierra el SEAM D1) ------------
# El tablero entero -- mapa "Criticidad Simulada", grafo reconstruido e "Importancia
# Variables" -- responde a este modelo y a esta unidad: la BOLSA (vano x ventana), que es
# la unidad en la que 04 define la criticidad. MGCECDL por fila salio del cuaderno: tener
# dos modelos contestando paneles vecinos del mismo tablero significaba que el panel y el
# mapa hablaban de cosas distintas sin que nada en pantalla lo dijera.
# Requiere DOS artefactos que produce el cuaderno 05 y que viven bajo `data/` (ignorado
# por git): el modelo y el cache de bolsas. Falla AQUI, con el nombre del cuaderno que los
# genera, en vez de a los diez minutos en la celda del boton.
RUTA_MODELO_MIL = MODEL_DIR / 'mil_vano_ventana_v1.pt'
RUTA_BOLSAS_MIL = ROOT / 'data' / 'derived' / 'bolsas_mil_full.joblib'
for _ruta in (RUTA_MODELO_MIL, RUTA_BOLSAS_MIL):
    assert _ruta.exists(), (
        f'Falta {_ruta.name}: lo produce 05_mil_vano_ventana.ipynb. Corre ese cuaderno '
        'antes que este.'
    )

BOLSAS = cargar_bolsas(RUTA_BOLSAS_MIL)
X_INST, FEATURES_MIL, BAG_INDEX = BOLSAS['X'], BOLSAS['features'], BOLSAS['bag_index']
# `device='cpu'` a proposito y no DEVICE: una seleccion son decenas de instancias, asi que
# el traslado a GPU/MPS cuesta mas de lo que ahorra, y saca una variable de dtype de en
# medio de un camino interactivo.
MIL = cargar_modelo_mil(RUTA_MODELO_MIL, device='cpu', features_esperadas=FEATURES_MIL)

# La guarda que hace comparables los dos mapas: si el MIL se hubiera entrenado con OTRA
# geometria KMeans, sus clases usarian los mismos 4 colores para significar otra cosa.
for _campo in ('offset', 'scale', 'centroides'):
    assert np.allclose(getattr(MIL.geometria, _campo), getattr(GEOMETRIA_014, _campo)), (
        f'La geometria del modelo MIL difiere de la de 01.4 en {_campo}: sus clases NO son '
        'las del mapa base y no pueden compartir la paleta.'
    )
assert tuple(MIL.geometria.logs) == tuple(GEOMETRIA_014.logs)

# Las 70 primeras features del MIL son exactamente las de MGCECDL (22 estaticas + 48 de
# clima); las 10 restantes son COD_CAUSA y sus indicadores, que no son controles del
# simulador. Por eso el catalogo de knobs de la celda siguiente sirve para los dos.
assert list(FEATURES_MIL[:len(feature_names)]) == list(feature_names), (
    'Las features del MIL ya no empiezan por las de MGCECDL: el catalogo de knobs '
    'apuntaria a columnas equivocadas.'
)
# Los MODOS de variable con los que el modelo agrupa las columnas -- son los mismos que
# usa la fusion FiLM (el clima reescala lo estructural), asi que colorear los nodos del
# grafo por modalidad muestra exactamente la particion que el modelo usa por dentro.
COLUMNAS_MODALIDAD = {m: set(int(i) for i in idx)
                      for m, idx in MIL.model.base.modality_feature_indices.items()}
MODALIDADES_MIL = list(COLUMNAS_MODALIDAD)
assert len(MODALIDADES_MIL) == len(PALETA_MODALIDADES), (
    f'El artefacto trae {len(MODALIDADES_MIL)} modalidades y la figura tiene trazas de '
    f'nodo para {len(PALETA_MODALIDADES)}: agrega la traza que falta antes de seguir.'
)
COLORES_MODALIDAD = dict(zip(MODALIDADES_MIL, PALETA_MODALIDADES))

print(f'MIL cargado -- {len(BAG_INDEX.keys):,} bolsas | {X_INST.shape[0]:,} instancias x '
      f'{len(FEATURES_MIL)} features | geometria identica a 01.4')
print('modos de variable: ' + ' | '.join(
    f'{m} ({len(COLUMNAS_MODALIDAD[m])})' for m in MODALIDADES_MIL))

In [ ]:
# --- construir_ventanas + per-(vano, ventana) events + caches (design section A) -------
VENTANAS = construir_ventanas(context_df['FECHA'])
TABLA = construir_tabla_vano_ventana(context_df, VENTANAS)
mask_para = construir_mask_cache(TABLA)
clases_para = construir_hist_class_cache(TABLA, mask_para)

# La clase de CADA celda (vano x ventana) de una sola pasada: es la misma asignacion por
# centroide mas cercano que hace `clases_para` ventana por ventana (es puntual, fila a
# fila), pero calculada una vez para poder dibujar la nube KMeans completa de la fila 3.
CLASE_TABLA, _n_clamped = cargar_clases_desde_014(
    TABLA['num_eventos'].to_numpy(dtype=float),
    TABLA['uiti_acumulado'].to_numpy(dtype=float),
)
# La nube de fondo va SUBMUESTREADA (ver `nube_fondo`): las 111 mil celdas completas son
# 1,2 MB de coordenadas en una sola rafaga por el comm del widget, por encima del
# `iopub_data_rate_limit` de 1 MB/s que ipykernel trae por defecto -- y un mensaje que se
# pasa de ese limite se descarta, con lo que la figura no llega a dibujarse.
NUBE_FONDO = nube_fondo(TABLA, CLASE_TABLA)
# Extension FIJA del plano (eventos, UITI), como en 01.4: los ejes y la frontera no
# dependen de la seleccion, solo del dataset.
EXTENSION = [float(TABLA['num_eventos'].min()), float(TABLA['num_eventos'].max()),
             float(TABLA['uiti_acumulado'].min()), float(TABLA['uiti_acumulado'].max())]
_n_nube = sum(len(c['x']) for c in NUBE_FONDO)
print(f'nube KMeans: {len(TABLA):,} celdas | {_n_nube:,} dibujadas (muestra fija) | '
      f'por clase {[len(c["x"]) for c in NUBE_FONDO]} | '
      f'{_n_clamped} valores recortados por eps')

CIRCUITOS = sorted(TABLA['CIRCUITO'].astype(str).unique())
VANOS_POR_CIRCUITO = {
    c: sorted(g['FID_VANO'].unique().tolist())
    for c, g in TABLA.groupby(TABLA['CIRCUITO'].astype(str))
}

print(f'{len(TABLA):,} celdas vano x ventana con eventos | {len(VENTANAS)} ventanas | '
      f'{TABLA["FID_VANO"].nunique():,} vanos distintos | {len(CIRCUITOS)} circuitos')


# Geometria FISICA de cada vano (no confundir con la geometria KMeans de la celda 4): mismo
# shapefile y mismo join que el mapa de 01.3/01.4. No se extrae a src/ porque es solo
# lectura + reindexado geoespacial, sin logica propia que valga la pena testear por fuera
# de lo que TABLA/capas_mapa_historico ya cubren.
def _norm_id(serie):
    return (serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .replace({'': pd.NA, '<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA}))


_lineas = gpd.read_file(ROOT / 'data' / 'GEO' / 'MVLINSEC.shp')
if str(_lineas.crs) != 'EPSG:4326':
    _lineas = _lineas.to_crs('EPSG:4326')
_lineas['FID_VANO_GEO'] = _norm_id(_lineas['G3E_FID'])
_utiles = _lineas[_lineas['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]

GEO_POR_CIRCUITO = {}
for _c, _g in _utiles.groupby(_utiles['CIRCUITO'].astype(str)):
    fids, lats, lons = [], [], []
    for _fid, _geom in zip(_g['FID_VANO_GEO'], _g.geometry):
        if _geom is None or _geom.is_empty:
            continue
        for _p in ([_geom] if _geom.geom_type == 'LineString' else list(getattr(_geom, 'geoms', []))):
            xs, ys = _p.xy
            fids.append(str(_fid))
            lats.append([round(v, 5) for v in ys])
            lons.append([round(v, 5) for v in xs])
    if fids:
        # `bounds` es lo que permite encuadrar el mapa sobre el circuito elegido, igual
        # que 01.4: [lat_min, lat_max, lon_min, lon_max].
        _la = [v for l in lats for v in l]
        _lo = [v for l in lons for v in l]
        GEO_POR_CIRCUITO[_c] = {
            'fids': fids, 'lat': lats, 'lon': lons,
            'bounds': [round(min(_la), 5), round(max(_la), 5),
                       round(min(_lo), 5), round(max(_lo), 5)],
        }


def _equipo(nombre):
    """Transformadores e interruptores del circuito, igual que 01.4 celda 5. Si el
    shapefile no esta, el mapa se dibuja sin equipos en vez de fallar: son contexto
    de lectura, no el dato del tablero."""
    ruta = ROOT / 'data' / 'GEO' / nombre
    if not ruta.exists():
        return {}
    g = gpd.read_file(ruta)
    if str(g.crs) != 'EPSG:4326':
        g = g.to_crs('EPSG:4326')
    g = g[g['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    return {c: {'lat': [round(float(p.y), 5) for p in gg.geometry],
                'lon': [round(float(p.x), 5) for p in gg.geometry]}
            for c, gg in g.groupby(g['CIRCUITO'].astype(str))}


TRAFOS = _equipo('GDBCHEC_TRANSFOR.shp')
SWITCHES = _equipo('SWITCHES.shp')

# UITI y eventos por vano y ventana: solo alimentan el hover, igual que 01.4. El grupo
# NO se guarda aqui -- sale de `clases_para`, que es la unica fuente de clases.
DATOS_VENTANA = [{} for _ in VENTANAS]
for _fid, _vi, _u, _n in zip(TABLA['FID_VANO'], TABLA['ventana_i'],
                             TABLA['uiti_acumulado'], TABLA['num_eventos']):
    DATOS_VENTANA[int(_vi)][str(_fid)] = (float(_u), int(_n))

print(f'{len(GEO_POR_CIRCUITO)} circuitos con geometria fisica | '
      f'{sum(len(v["lat"]) for v in TRAFOS.values()):,} transformadores | '
      f'{sum(len(v["lat"]) for v in SWITCHES.values()):,} switches')

In [ ]:
KNOBS = build_knobs(
    feature_names=feature_names,
    original_feature_df=Xdf,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)
print(f'{len(KNOBS)} controles (Knob catalog, PR2a) -- '
      f'{sum(1 for k in KNOBS if k.kind == "categorical")} categoricos, '
      f'{sum(1 for k in KNOBS if k.kind == "numeric")} numericos, '
      f'{sum(1 for k in KNOBS if k.kind == "constant")} constantes')

In [ ]:
# --- Inventario de trazas CONGELADO (design section G). Los indices 0-17 no se mueven:
# la grilla cambio de 2x3 a 3x2 y ninguna traza cambio de posicion en la LISTA, solo de
# subplot. El grafo reconstruido (decision D4) sigue mutando el indice 7 en un PR futuro,
# con sus trazas nuevas a partir del 18.
# Los mapas ocupan las DOS columnas de su fila (colspan=2): un mapa geografico compartido
# entre 4 filas de vanos y una barra de importancia no se lee, y estirarlo al ancho
# completo es lo unico que permite distinguir tramos vecinos. Las otras dos figuras --
# importancia de variables y el panel reservado -- bajan juntas a la fila 3.
# Los equipos (14-17) van ultimos por orden de dibujo: los marcadores tienen que quedar
# por encima de las lineas, como en 01.4.
IDX = {
    'clases': [0, 1, 2, 3],          # fila 1 (2 columnas) -- mapa historico (01.4), PR3
    'sin_dato': 4,                    # fila 1 -- sin eventos en la ventana
    'marcados': 5,                    # fila 1 -- halo de vanos marcados
    'ranking': 6,                     # fila 3 col 1 -- importancia de variables, PR4
    'grafo_aristas': 7,               # fila 3 col 3 -- grafo reconstruido (decision D4)
    'pred_clases': [8, 9, 10, 11],    # fila 2 (2 columnas) -- mapa predicho MGCECDL, PR5
    'pred_sin_dato': 12,               # fila 2, PR5
    'pred_marcados': 13,               # fila 2, PR5
    'trafos': 14,                      # fila 1 -- equipos, PR6
    'switches': 15,                    # fila 1 -- equipos, PR6
    'pred_trafos': 16,                 # fila 2 -- equipos, PR6
    'pred_switches': 17,               # fila 2 -- equipos, PR6
    # Nuevas, a partir del 18 y sin mover ninguna anterior (design section G).
    'marcados_clases': [18, 19, 20, 21],  # fila 1 -- marcado, con el color de SU clase
    'marcados_sin_dato': 22,              # fila 1 -- marcado sin celda en la ventana: negro
    'nube_clases': [23, 24, 25, 26],      # fila 3 col 2 -- nube KMeans (fondo fijo)
    'nube_seleccion': 27,                 # fila 3 col 2 -- celdas de lo marcado
    'grafo_pesos': 28,                    # fila 3 col 3 -- peso de cada arista
    'grafo_nodos': [29, 45],              # fila 3 col 3 -- variables, una traza por modo
    'frontera': 30,                       # fila 3 col 2 -- Voronoi de las fronteras KMeans
    'evolucion': [31, 32, 33, 34, 35, 36],  # fila 4 col 1 -- una serie por vano marcado
    'violin_uiti': [37, 38, 39, 40],      # fila 4 col 2 -- reparto de UITI por grupo
    'violin_eventos': [41, 42, 43, 44],   # fila 4 col 3 -- reparto de eventos por grupo
}

_fig = make_subplots(
    rows=4, cols=3,
    specs=[[{'type': 'map', 'colspan': 3}, None, None],
           [{'type': 'map', 'colspan': 3}, None, None],
           [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}],
           [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}]],
    # Un titulo por subplot REAL: las celdas `None` del colspan no consumen ninguno.
    subplot_titles=(
        'Criticidad Original',
        'Criticidad Simulada',
        'Importancia Variables',
        'Grupos KMeans de vanos',
        'Grafo reconstruido',
        'Evolucion temporal de los vanos marcados',
        'UITI por grupo',
        'Eventos por grupo',
    ),
    row_heights=[0.29, 0.29, 0.21, 0.21],
    horizontal_spacing=0.07, vertical_spacing=0.06,
)

for _clase in range(4):                                          # 0-3
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='hist', legendgrouptitle_text='Criticidad original',
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 4
    lat=[], lon=[], mode='lines', name='Sin evento en la ventana', legendgroup='hist',
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)
# El marcado de la fila 1 va en DOS capas, como en 01.4: primero el halo blanco ancho
# (esta traza, la 5), y despues -- indices 18-22, al final por orden de dibujo -- la linea
# con el color de la clase del vano. El halo no va a la leyenda: una linea blanca sobre
# fondo blanco no dice nada ahi.
_fig.add_trace(go.Scattermap(                                    # 5
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='hist',
    showlegend=False,
    line=dict(width=ANCHO_HALO, color=COLOR_HALO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)

_fig.add_trace(go.Bar(x=[], y=[], orientation='h', showlegend=False,
                      hovertext=[], hoverinfo='text'), row=3, col=1)  # 6
# La barra ya no muestra la magnitud cruda sino su participacion softmax: un porcentaje
# se lee sin conocer las unidades del modelo, una "sensibilidad min-max" de 0.0143 no.
_fig.update_xaxes(title_text='Relevancia (softmax)', tickformat='.0%', row=3, col=1)
# `type='category'`: con la traza vacia un eje numerico inventa marcas 0..4 que no
# significan nada. Un eje de categorias vacio no dibuja ninguna.
_fig.update_yaxes(type='category', tickfont=dict(size=10), row=3, col=1)

# La 7 era la traza RESERVADA para el grafo reconstruido (decision D4). Este PR la puebla:
# pasa a ser el trazo de las aristas. Su indice nunca se movio, que era el punto de
# haberla dejado ahi desde el principio.
_fig.add_trace(go.Scatter(
    x=[], y=[], mode='lines', showlegend=False,
    line=dict(width=1.0, color='rgba(120,110,110,0.45)'), hoverinfo='skip',
), row=3, col=3)  # 7

for _clase in range(4):                                          # 8-11
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='pred', legendgrouptitle_text='Criticidad simulada', showlegend=False,
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 12
    lat=[], lon=[], mode='lines', name='Sin evento / no simulado', legendgroup='pred',
    showlegend=False, line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 13
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='pred', showlegend=False,
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_MARCADO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)

# Los equipos van al final para dibujarse ENCIMA de los tramos. Se repiten por fila
# porque una traza pertenece a un solo subplot: no hay forma de compartirla entre los
# dos mapas, y sin ellos la fila 2 se leeria como otra geografia.
for _fila, _leyenda in ((1, True), (2, False)):
    for _nombre, _color, _tam in [('Transformadores', COLOR_TRAFO, 6),
                                  ('Switches', COLOR_SWITCH, 5)]:
        _fig.add_trace(go.Scattermap(                             # 14-17
            lat=[], lon=[], mode='markers', name=_nombre,
            legendgroup='equipos', legendgrouptitle_text='Equipos',
            showlegend=_leyenda,
            marker=dict(size=_tam, color=_color), hovertext=[], hoverinfo='text',
        ), row=_fila, col=1)

# --- 18-22: el marcado de la fila 1, con el color de su clase (paridad 01.4) ---------
for _clase in range(4):                                          # 18-21
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='hist', showlegend=False,
        line=dict(width=ANCHO_MAPA_MARCADO, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 22
    lat=[], lon=[], mode='lines', name='Marcado sin eventos', legendgroup='hist',
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)

# --- 23-27: la nube KMeans de 01.4 en el panel que estaba reservado ------------------
# El fondo son TODAS las celdas (vano x ventana) del dataset, agrupadas por clase: es
# donde estan las fronteras, y no se mueve nunca. Encima, las celdas de lo marcado en la
# ventana activa. Ahi se ve lo que el mapa solo insinua: mover la ventana mueve el punto
# del vano por el plano (eventos, UITI) y por eso cambia su clase.
for _clase in range(4):                                          # 23-26
    _fig.add_trace(go.Scattergl(
        x=[], y=[], mode='markers', name=NOMBRES_GRUPOS[_clase],
        legendgroup='nube', legendgrouptitle_text='Nube KMeans', showlegend=False,
        marker=dict(size=3.5, color=COLORES_GRUPOS[_clase], opacity=OPACIDAD_NUBE),
        hoverinfo='skip',
    ), row=3, col=2)
_fig.add_trace(go.Scattergl(                                     # 27
    x=[], y=[], mode='markers', name='Seleccion', showlegend=False,
    marker=dict(size=9, color=[], line=dict(width=1.4, color='#111111')),
    hovertext=[], hoverinfo='text',
), row=3, col=2)
_fig.update_xaxes(title_text='Eventos en la ventana',
                  type='log' if GEOMETRIA_014.logs[0] else 'linear', row=3, col=2)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear', row=3, col=2)

# --- 28-29: el grafo reconstruido de la seleccion (decision D4) ----------------------
# El peso viaja en un marcador en el PUNTO MEDIO de cada arista y no en el ancho de la
# linea: una sola traza de lineas no puede variar su ancho por segmento, y partirla en
# una traza por arista serian 64 trazas que hay que restilar una por una.
_fig.add_trace(go.Scattergl(                                     # 28
    x=[], y=[], mode='markers', showlegend=False,
    marker=dict(size=[], color=[], colorscale='Reds', cmin=0.0, showscale=False,
                line=dict(width=0.4, color='#5b4a48')),
    hovertext=[], hoverinfo='text',
), row=3, col=3)
# Los nodos van en UNA traza por modo de variable, no en una sola con vector de colores:
# asi cada modo entra por si mismo a la leyenda, que es donde se lee que significa el
# color. La segunda traza (45) se crea al final del inventario, para no correr ningun
# indice existente -- por eso hay un constructor compartido y dos llamadas separadas.
def _traza_nodos_grafo(modalidad):
    return go.Scatter(
        x=[], y=[], mode='markers+text', name=modalidad,
        legendgroup='grafo', legendgrouptitle_text='Modo de variable',
        marker=dict(size=7, color=COLORES_MODALIDAD[modalidad],
                    line=dict(width=0.5, color='#1f2937')),
        text=[], textposition='middle right', textfont=dict(size=7, color='#334155'),
        hovertext=[], hoverinfo='text',
    )


_fig.add_trace(_traza_nodos_grafo(MODALIDADES_MIL[0]), row=3, col=3)   # 29
# Sin ejes: un grafo en disposicion circular no mide nada en x ni en y. El rango se fija a
# mano y con holgura -- sin ella los rotulos de los nodos del borde salen cortados.
_fig.update_xaxes(visible=False, showticklabels=False, range=[-1.75, 1.75], row=3, col=3)
_fig.update_yaxes(visible=False, showticklabels=False, range=[-1.3, 1.3], row=3, col=3)
_fig.add_annotation(text='', xref='x3 domain', yref='y3 domain', x=0.5, y=0.5,
                    showarrow=False, align='center',
                    font=dict(size=11, color='#7a5c58'))
IDX_ANOTACION_GRAFO = len(_fig.layout.annotations) - 1

# --- 30: frontera KMeans (Voronoi) DEBAJO de la nube --------------------------------
# Misma escala escalonada y misma opacidad de 01.4. Va como `Contour` y no como trazas
# de linea porque Plotly dibuja contornos en una capa POR DEBAJO de los scatter del mismo
# subplot, sin importar el orden de las trazas: la nube queda encima aunque esta traza sea
# la 30.
ESCALA_CONTORNO = []
for _g, _color in enumerate(COLORES_GRUPOS):
    ESCALA_CONTORNO.append([_g / 4.0, _color])
    ESCALA_CONTORNO.append([(_g + 1) / 4.0, _color])
_fig.add_trace(go.Contour(                                       # 30
    z=[[0, 0], [0, 0]], x=[0, 1], y=[0, 1], colorscale=ESCALA_CONTORNO,
    zmin=-0.5, zmax=3.5, showscale=False, opacity=OPACIDAD_FRONTERA, hoverinfo='skip',
    line=dict(width=1.2, color='rgba(120,20,20,0.6)'),
    contours=dict(start=-0.5, end=3.5, size=1, coloring='fill'), showlegend=False,
), row=3, col=2)

# --- 31-36: evolucion temporal, un CUPO por vano marcado ----------------------------
# Cupos fijos y no una traza por vano: el inventario de trazas esta congelado, y un
# circuito puede tener cientos de vanos marcados. Se dibujan los primeros
# N_CUPOS_EVOLUCION y el panel dice cuantos quedaron afuera, igual que 01.4.
for _cupo in range(N_CUPOS_EVOLUCION):                           # 31-36
    _fig.add_trace(go.Scatter(
        x=[], y=[], mode='lines+markers', name='', showlegend=False,
        line=dict(color=COLORES_VANOS[_cupo], width=2), marker=dict(size=5),
        connectgaps=False,  # el hueco de una ventana sin celda NO se cose
        hovertext=[], hoverinfo='text',
    ), row=4, col=1)
_fig.update_xaxes(title_text='Ventana', tickmode='array',
                  tickvals=[v['i'] for v in VENTANAS],
                  ticktext=[v['etiqueta'] for v in VENTANAS],
                  tickfont=dict(size=9), row=4, col=1)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear', row=4, col=1)

# --- 37-44: violines por grupo (UITI y eventos) -------------------------------------
for _fila_violin, _columna, _campo in ((37, 2, 'UITI'), (41, 3, 'Eventos')):
    for _clase in range(4):                                      # 37-40 y 41-44
        _fig.add_trace(go.Violin(
            y=[], name=NOMBRES_GRUPOS[_clase], showlegend=False,
            fillcolor=COLORES_GRUPOS[_clase], opacity=0.85,
            line=dict(color='#5b4a48', width=1),
            box_visible=True, meanline_visible=False, points=False, spanmode='hard',
            hovertemplate=f'{_campo} -- %{{x}}: %{{y:,.2f}}<extra></extra>',
        ), row=4, col=_columna)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear', row=4, col=2)
_fig.update_yaxes(title_text='Eventos', rangemode='tozero', row=4, col=3)

_fig.add_trace(_traza_nodos_grafo(MODALIDADES_MIL[1]), row=3, col=3)   # 45
_fig.update_xaxes(tickfont=dict(size=9), row=4, col=2)
_fig.update_xaxes(tickfont=dict(size=9), row=4, col=3)

# El aviso del mapa simulado va en coordenadas de PAPEL y no de eje: un subplot de tipo
# `map` no tiene ejes cartesianos a los que anclar una anotacion. El centro sale del
# dominio que `make_subplots` ya calculo, asi cambiar `row_heights` no lo desalinea.
_dominio_simulado = _fig.layout.map2.domain
_fig.add_annotation(
    text='', xref='paper', yref='paper',
    x=(_dominio_simulado.x[0] + _dominio_simulado.x[1]) / 2.0,
    y=(_dominio_simulado.y[0] + _dominio_simulado.y[1]) / 2.0,
    showarrow=False, align='center', font=dict(size=13, color='#5b4a48'),
    bgcolor='rgba(255,255,255,0.88)', bordercolor='#e4c4c0', borderwidth=1, borderpad=8,
)
IDX_ANOTACION_SIMULADO = len(_fig.layout.annotations) - 1

_fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    map2=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    title=dict(text='Simulador Criticidad'),
    # Dos mapas apilados a ancho completo piden alto: con los 760 de la grilla 2x3 cada
    # mapa quedaba en una franja de ~300 px y los tramos se pisaban entre si.
    # Ancho PROPIO y no `autosize`. La grilla 4x3 esta dimensionada para estos 1280 px:
    # atada al ancho de la celda -- casi siempre menor -- los dos mapas quedan angostos y
    # las seis casillas de las filas 3 y 4 se aprietan hasta pisarse los rotulos. El modo
    # responsive pertenece al panel WEB, que se exporta con `width=None` sobre una copia
    # de esta figura, no a la celda del cuaderno.
    height=1560, width=1280, template='plotly_white',
    legend=dict(y=1.0, yanchor='top'),
)

# Los indices se verifican al generar, igual que 01.3/01.4: si alguien reordena las
# trazas esto falla AQUI, no se descubre silenciosamente en la celda de dibujo.
assert len(_fig.data) == 46, len(_fig.data)
assert _fig.layout.width and _fig.layout.height, (
    'la figura de la CELDA lleva ancho y alto propios; el panel web se exporta sobre una '
    'copia con `width=None`, que es donde vive el modo responsive')
assert all(_fig.data[i].type == 'scattermap' for i in IDX['clases'] + [IDX['sin_dato'], IDX['marcados']])
assert [_fig.data[i].line.color for i in IDX['clases']] == COLORES_GRUPOS
assert _fig.data[IDX['ranking']].type == 'bar'
assert _fig.data[IDX['grafo_aristas']].type == 'scatter'
assert _fig.data[IDX['grafo_aristas']].mode == 'lines'
assert all(_fig.data[i].type == 'scattermap'
           for i in IDX['pred_clases'] + [IDX['pred_sin_dato'], IDX['pred_marcados']])
# Los equipos son PUNTOS y van despues de todas las lineas: si alguien los adelanta,
# quedan tapados por los tramos y esto falla al generar, no en el navegador.
assert all(_fig.data[i].mode == 'markers'
           for i in (IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']))
assert min(IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']) > IDX['pred_marcados']
# El marcado con color de clase va DESPUES del halo blanco, o el halo lo taparia.
assert min(IDX['marcados_clases']) > IDX['marcados']
assert [_fig.data[i].line.color for i in IDX['marcados_clases']] == COLORES_GRUPOS
assert _fig.data[IDX['marcados_sin_dato']].line.color == COLOR_SIN_EVENTO
# El MISMO negro para la ausencia en los dos mapas: si alguien los separa, esto falla al
# generar la figura y no en la lectura de un tablero ya publicado.
assert (_fig.data[IDX['sin_dato']].line.color
        == _fig.data[IDX['pred_sin_dato']].line.color == COLOR_SIN_EVENTO)
assert all(_fig.data[i].type == 'scattergl' for i in IDX['nube_clases'] + [IDX['nube_seleccion']])
assert _fig.data[IDX['frontera']].type == 'contour'
assert all(_fig.data[i].type == 'violin' for i in IDX['violin_uiti'] + IDX['violin_eventos'])
assert [_fig.data[i].fillcolor for i in IDX['violin_uiti']] == COLORES_GRUPOS
assert len(IDX['evolucion']) == N_CUPOS_EVOLUCION
assert [_fig.data[i].name for i in IDX['grafo_nodos']] == MODALIDADES_MIL
assert ([_fig.data[i].marker.color for i in IDX['grafo_nodos']]
        == [COLORES_MODALIDAD[m] for m in MODALIDADES_MIL])

fig = go.FigureWidget(_fig)
print(f'FigureWidget con {len(fig.data)} trazas (indices 0-13 congelados, design section G)')

In [ ]:
# --- Fila 1: mapa historico con paridad 01.4 + seleccion por casilla o por clic ------
# Tres cosas que el mapa de 01.4 hace y este no hacia: se ENCUADRA sobre el circuito
# elegido (sin eso el circuito queda como un garabato diminuto en un mapa centrado en
# Manizales), dibuja transformadores e interruptores, y da hover por tramo. La cuarta es
# la seleccion: en 01.4 un vano se marca con su casilla O tocandolo en el mapa, y las dos
# vias son EL MISMO estado -- el clic alterna la casilla y deja que todo se rehaga desde
# ahi. Un registro paralelo es como la lista, el mapa y el ranking empiezan a contar
# cosas distintas.


def _seleccion_actual():
    return circuito_widget.value, ventana_widget.value, set(vano_widget.value)


def _capas_de_la_seleccion(clases_por_fid, *, campo, nombres_clase):
    """Las capas de UN mapa, con las etiquetas y el customdata que necesita el clic.

    `campo` nombra en el tooltip a que pertenece la clase -- "Criticidad original" en
    la fila 1, "Criticidad simulada" en la fila 2. Esa distincion vive ahora en el tooltip de
    cada tramo y en la leyenda, que es donde se lee mientras se mira el mapa, en vez de
    en un parrafo fijo al costado del panel.
    """
    circuito, ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    ventana = VENTANAS[ventana_i]
    datos = DATOS_VENTANA[ventana_i]

    etiquetas = {}
    for fid in geo['fids']:
        uiti, eventos = datos.get(fid, (0.0, 0))
        clase = clases_por_fid.get(fid)
        # Sin celda en la ventana no hay clase, y eso NO es el grupo mas bajo: es la
        # ausencia del dato. Mismo criterio que el tooltip de 01.4.
        etiquetas[fid] = (
            f'<b>Vano {fid}</b><br>{ventana["etiqueta"]}: {ventana["periodo"]}'
            f'<br>{campo}: {nombres_clase[clase] if clase is not None else "sin dato"}'
            f'<br>UITI acumulado: {uiti}<br>Eventos: {eventos}'
            + ('<br>(marcado)' if fid in marcados else '')
        )
    # `marca_extremos` agrega el guion horizontal de inicio y fin de cada vano (01).
    # Va con etiqueta VACIA en sus puntos -- el vertice real del extremo esta en el centro
    # del guion y ya la lleva -- para no triplicar el hovertext que viaja por el comm.
    return capas_mapa_historico(geo, clases_por_fid, marcados=marcados,
                                etiquetas_por_fid=etiquetas, marca_extremos=MARCA_VANO)


def _volcar_capa(traza, capa):
    """Las cuatro columnas van juntas SIEMPRE: si `customdata` se desfasa de lat/lon,
    Plotly desalinea el resto de la traza y el clic devuelve el vano equivocado."""
    traza.lat = capa['lat']
    traza.lon = capa['lon']
    traza.hovertext = capa['hovertext']
    traza.customdata = capa['customdata']


def _redibujar_mapa_historico(*_ignorado):
    circuito, ventana_i, _marcados = _seleccion_actual()
    capas = _capas_de_la_seleccion(clases_para(circuito, ventana_i),
                                   campo='Criticidad original', nombres_clase=NOMBRES_GRUPOS)
    mask_ventana = mask_para(circuito, ventana_i)
    seleccion = nube_seleccion(TABLA, CLASE_TABLA, mask_ventana=mask_ventana,
                               marcados=_marcados)
    # Los violines describen SOLO los vanos marcados (regla de 01.4, ver
    # `reparto_por_clase`); la evolucion, los primeros N_CUPOS_EVOLUCION de ellos.
    reparto = reparto_por_clase(TABLA, CLASE_TABLA, mask_ventana=mask_ventana,
                                marcados=_marcados)
    marcados_ordenados = [f for f in GEO_POR_CIRCUITO.get(circuito, {}).get('fids', [])
                          if f in _marcados]
    marcados_ordenados = list(dict.fromkeys(marcados_ordenados))[:N_CUPOS_EVOLUCION]
    series = series_temporal_vanos(TABLA, circuito=circuito, fids=marcados_ordenados,
                                   n_ventanas=len(VENTANAS))
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['clases'][_clase]], capas['clases'][_clase])
            _volcar_capa(fig.data[IDX['marcados_clases'][_clase]],
                         capas['marcados_por_clase'][_clase])
        _volcar_capa(fig.data[IDX['sin_dato']], capas['sin_dato'])
        _volcar_capa(fig.data[IDX['marcados']], capas['marcados'])
        _volcar_capa(fig.data[IDX['marcados_sin_dato']], capas['marcados_sin_dato'])
        # La nube: solo el resaltado se repinta, el fondo se dibuja una vez al arrancar.
        _traza_nube = fig.data[IDX['nube_seleccion']]
        _traza_nube.x = seleccion['x']
        _traza_nube.y = seleccion['y']
        _traza_nube.marker.color = [COLORES_GRUPOS[c] for c in seleccion['clase']]
        _traza_nube.hovertext = [
            f'<b>Vano {f}</b><br>Eventos: {x:,}<br>UITI acumulado: {y}'
            f'<br>Grupo: {NOMBRES_GRUPOS[c]}'
            for f, x, y, c in zip(seleccion['fid'], seleccion['x'],
                                  seleccion['y'], seleccion['clase'])
        ]
        # Fila 4: evolucion por cupo y violines por grupo.
        for _cupo, _traza in enumerate(fig.data[i] for i in IDX['evolucion']):
            _serie = series[_cupo] if _cupo < len(series) else None
            _traza.x = _serie['x'] if _serie else []
            _traza.y = _serie['uiti'] if _serie else []
            _traza.hovertext = ([
                f'<b>Vano {_serie["fid"]}</b><br>{VENTANAS[i]["etiqueta"]}: '
                f'{VENTANAS[i]["periodo"]}<br>UITI: {u}<br>Eventos: {e}'
                for i, u, e in zip(_serie['x'], _serie['uiti'], _serie['eventos'])
            ] if _serie else [])
        for _clase in range(4):
            fig.data[IDX['violin_uiti'][_clase]].y = reparto[_clase]['uiti']
            fig.data[IDX['violin_eventos'][_clase]].y = reparto[_clase]['eventos']
        # Mismo motivo que el eje de relevancia: sin vanos marcados el eje lineal de
        # eventos autoescala a [-1, 4] y muestra un "-1 eventos" que no existe.
        _max_eventos = max((e for g in reparto for e in g['eventos']), default=0)
        fig.update_yaxes(range=[0, _max_eventos * 1.15 if _max_eventos else 5],
                         row=4, col=3)


def _pintar_circuito(*_ignorado):
    """Lo que depende del CIRCUITO y no de la ventana: equipos y encuadre. Se separa del
    repintado por ventana porque mover la ventana no tiene por que recentrar el mapa --
    en 01.4 el encuadre tambien se hace una sola vez por circuito (`ULTIMO_CENTRADO`)."""
    circuito = circuito_widget.value
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    # El tamanio en pixeles del subplot de mapa, de su dominio por el de la figura.
    # Sin esto el zoom salia del span en GRADOS, sin mirar el viewport, y un circuito
    # alto quedaba recortado arriba y abajo. Aqui se conocen las dos dimensiones, asi
    # que el encuadre es exacto.
    _dom = fig.layout.map.domain
    vista = centro_y_zoom(
        GEO_POR_CIRCUITO.get(circuito, {}).get('bounds'),
        ancho_px=float(fig.layout.width) * float(_dom.x[1] - _dom.x[0]),
        alto_px=float(fig.layout.height) * float(_dom.y[1] - _dom.y[0]))
    with fig.batch_update():
        # Solo la fila 1: los equipos de la fila 2 los pinta el mapa simulado, que antes
        # de la primera simulacion no muestra NADA.
        for _i_tr, _i_sw in ((IDX['trafos'], IDX['switches']),):
            fig.data[_i_tr].lat, fig.data[_i_tr].lon = tr['lat'], tr['lon']
            fig.data[_i_tr].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
            fig.data[_i_sw].lat, fig.data[_i_sw].lon = sw['lat'], sw['lon']
            fig.data[_i_sw].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        if vista is not None:
            # Los dos mapas comparten encuadre a proposito: la comparacion fila 1 contra
            # fila 2 solo se sostiene si las dos miran exactamente la misma geografia.
            for _mapa in ('map', 'map2'):
                getattr(fig.layout, _mapa).center = vista['center']
                getattr(fig.layout, _mapa).zoom = vista['zoom']


_DESC = {'description_width': 'initial'}  # sin esto ipywidgets trunca los rotulos
circuito_widget = widgets.Dropdown(options=CIRCUITOS, description='Circuito',
                                   style=_DESC)
# El rotulo lleva las fechas del intervalo y no solo "V1": una ventana sin sus fechas
# obliga a ir a buscar a que periodo corresponde cada vez que se mueve el deslizador.
ventana_widget = widgets.SelectionSlider(
    options=[(f'{v["etiqueta"]}: {v["periodo"]}', v['i']) for v in VENTANAS],
    description='Ventana', continuous_update=False, style=_DESC,
    layout=widgets.Layout(width='560px'),
)
# Casillas, no SelectMultiple: es la unica forma de que un clic en el mapa alterne el
# MISMO control que el usuario ve, y de que marcar un vano no borre los ya marcados.
vano_widget = construir_selector_vanos(VANOS_POR_CIRCUITO.get(circuito_widget.value, []))


# "Marcar todos" / "Desmarcar", igual que el par de botones de 04: con circuitos de
# cientos de vanos, marcarlos de a uno no es una opcion. Los dos van por el selector y no
# por un registro propio, asi que emiten UN solo cambio de `value` y disparan un solo
# repintado -- no uno por casilla.
boton_marcar_todos = widgets.Button(description='Marcar todos', button_style='')
boton_desmarcar = widgets.Button(description='Desmarcar', button_style='')
boton_marcar_todos.on_click(lambda _b: vano_widget.marcar_todos())
boton_desmarcar.on_click(lambda _b: vano_widget.desmarcar_todos())


def _on_circuito_change(_change):
    vano_widget.poblar(VANOS_POR_CIRCUITO.get(circuito_widget.value, []))
    _pintar_circuito()
    _redibujar_mapa_historico()


def _al_hacer_clic(traza, puntos, _estado):
    """Un clic sobre un tramo alterna su vano. El fid sale de `customdata` y no del
    indice del punto: los tramos viajan concatenados con un `None` de separador, asi que
    ese indice cambia con la ventana."""
    fid = fid_de_punto(traza.customdata, getattr(puntos, 'point_inds', ()) or ())
    if fid is not None:
        vano_widget.alternar(fid)


# SOLO el mapa base. La fila 2 es la SALIDA del modelo, no un control: marcar un vano
# desde ahi mezcla "lo que yo elegi" con "lo que el modelo predijo" sobre la misma
# superficie, que es justo la confusion que separa a las dos filas (D2).
# Nota sobre el alcance del clic: plotly solo convierte un clic en evento si en ese punto
# hay hover, y en un `scattermap` de lineas el hover se calcula contra los VERTICES del
# tramo (`scattermap/hover.js`: distancia por punto, radio minimo 3 px, tope
# `layout.hoverdistance`). Hay que tocar el tramo cerca de uno de sus quiebres, no en
# cualquier parte del segmento. `hoverdistance` sube de los 20 px por defecto a 30 para
# que el blanco sea mas generoso sin llegar a marcar un vano lejano.
for _i_traza in IDX['clases'] + [IDX['sin_dato'], IDX['marcados']]:
    fig.data[_i_traza].on_click(_al_hacer_clic)
fig.layout.hoverdistance = 30

# Tier 0 del presupuesto de interactividad (design section A): elegir circuito, mover la
# ventana o marcar un vano no llama al modelo -- sin debounce ni epoch guard, que
# pertenecen al tier 1/2 (fila 2, ranking, boton "Simular"), fuera del alcance de este PR.
circuito_widget.observe(_on_circuito_change, names='value')
ventana_widget.observe(_redibujar_mapa_historico, names='value')
vano_widget.observe(_redibujar_mapa_historico, names='value')

# El fondo de la nube y la frontera van una sola vez: no dependen de la seleccion (01.4
# ajusta el KMeans una vez y elegir circuito o vanos solo cambia que se resalta).
FRONTERA = frontera_kmeans(GEOMETRIA_014, x_min=EXTENSION[0], x_max=EXTENSION[1],
                           y_min=EXTENSION[2], y_max=EXTENSION[3])
with fig.batch_update():
    for _clase in range(4):
        fig.data[IDX['nube_clases'][_clase]].x = NUBE_FONDO[_clase]['x']
        fig.data[IDX['nube_clases'][_clase]].y = NUBE_FONDO[_clase]['y']
    fig.data[IDX['frontera']].x = FRONTERA['x']
    fig.data[IDX['frontera']].y = FRONTERA['y']
    fig.data[IDX['frontera']].z = FRONTERA['z']

_pintar_circuito()               # equipos y encuadre del circuito inicial
_redibujar_mapa_historico()      # primer dibujo, con la seleccion inicial

In [ ]:
# --- Importancia de variables, fila 3 col 1 (design section A, decision D7) ------------
# Barrido de sensibilidad min-max sobre el MISMO modelo y la MISMA unidad que el mapa
# simulado: la bolsa (vano, ventana) del cuaderno 05. Antes corria sobre MGCECDL por
# fila, y entonces el panel y el mapa hablaban de cosas distintas -- el panel media el
# efecto de una variable sobre filas de evento sueltas, el mapa mostraba clases de bolsa
# -- sin que nada en pantalla lo dijera.
# Sigue sin ser SHAP (decision D5): es un barrido min-max, y asi se llama en todo el
# cuaderno. Corre UNA vez, dentro del mismo job del boton "Simular" (celda siguiente) y
# bajo la misma epoca: un solo disparador significa que mapa, grafo e importancia siempre
# describen la MISMA seleccion. Sin vanos marcados el grano es el circuito completo en esa
# ventana, las mismas bolsas que pinta el mapa.
from chec_local_interpreter.mil_simulador_015 import construir_relevance_cache_mil

rankear_relevancia = construir_relevance_cache_mil(
    predictor=MIL,
    X_inst=X_INST,
    bag_index=BAG_INDEX,
    feature_names=FEATURES_MIL,
    knobs=KNOBS,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)

RANKING_VACIO = {'vacio': True, 'filas': [], 'n_vanos': 0, 'n_filas': 0, 'mensaje': None}


def _calcular_ranking(circuito, ventana_i, marcados):
    """La parte pesada: `1 + 2 x knobs_numericos` pasadas de bolsas (una base compartida
    y el min/max de cada control). Se llama DENTRO del job de "Simular" para que una sola
    epoca cubra mapa, grafo e importancia. Medido sobre el modelo real: 0,09 s para 24
    bolsas y 0,34 s para 537, asi que alcanza con el LRU de sesion y no hace falta el
    cache en disco que necesitaba la version por filas."""
    return rankear_relevancia(circuito, VENTANAS[ventana_i]['etiqueta'], marcados)


def _pintar_ranking(resultado):
    """Repaint puro, cero pasadas del modelo."""
    # Orden ascendente: en un bar horizontal Plotly dibuja la primera categoria abajo, asi
    # que la variable mas relevante (primera en `filas`, ya ordenada descendente por
    # magnitud cruda) queda arriba.
    filas = list(reversed(resultado['filas']))
    # El rango se fija a mano: con la traza vacia Plotly autoescala a [-1, 4] y el
    # `tickformat` de porcentaje lo muestra como "0% ... 600%", que se lee como si algo
    # tuviera 600% de relevancia. Con datos se ajusta al maximo real, que ademas aprovecha
    # el ancho del panel sin tocar los valores.
    _tope = max((fila['relevancia'] for fila in filas), default=0.0)
    with fig.batch_update():
        fig.update_xaxes(range=[0, _tope * 1.15 if _tope > 0 else 0.25], row=3, col=1)
        fig.data[IDX['ranking']].y = [fila['label'] for fila in filas]
        fig.data[IDX['ranking']].x = [fila['relevancia'] for fila in filas]
        # La magnitud cruda no desaparece: baja al hover, que es donde se la consulta
        # cuando hace falta comparar contra otra corrida.
        fig.data[IDX['ranking']].hovertext = [
            f'<b>{fila["label"]}</b><br>Relevancia: {fila["relevancia"]:.1%}'
            f'<br>Sensibilidad min-max: {fila["magnitud_max_cambio_abs"]:.4g}'
            for fila in filas
        ]


_pintar_ranking(RANKING_VACIO)  # vacio hasta el primer "Simular"


In [ ]:
# --- Fila 2: mapa "Criticidad Simulada" + boton "Simular" (design section A, decision D2)
# El boton es el UNICO disparador y hace TRES cosas de una sola vez, bajo la misma epoca:
# el mapa simulado, el grafo reconstruido de la seleccion y el barrido de importancia de
# la celda anterior. Ya no hay alternador base/simulado/delta: el mapa de la fila 2
# muestra SIEMPRE la clase simulada, que es lo que el boton promete.
#
# El mapa y el grafo salen del modelo MIL del cuaderno 05, que puntua BOLSAS: una bolsa es
# una celda (vano, ventana) y su clase sale de `asignar_clase(n_obs OBSERVADO, u-hat
# predicho)` sobre la geometria KMeans de 01.4 -- la MISMA con la que se pinta el mapa
# base, que es por lo que los dos mapas comparten paleta por construccion y no por
# convencion. `n_obs` nunca se simula: es un eje del espacio que define la clase.
# Debounce asincronico (design section A): `asyncio.ensure_future` + cancelacion en el
# propio event loop del kernel, NUNCA `threading.Timer` -- ipykernel enruta la salida de
# los widgets con el parent header thread-local, asi que una escritura desde un hilo en
# segundo plano cae en la celda equivocada. `_EPOCA` es el guard de epoca: cualquier evento
# que invalide un job en vuelo la avanza y la escritura tardia se descarta.
from chec_local_interpreter.vano_app_015 import (
    DEBOUNCE_SEGUNDOS,
    ESTADO_SIMULADO,
    aplicar_si_vigente,
    clases_por_fid_para_estado,
    siguiente_epoca,
)
from chec_local_interpreter.vano_widgets import widget_for_knob

# El estado vacio se PIDE a la funcion en vez de escribirlo a mano: escrito a mano se
# desincroniza en cuanto `trazas_grafo` agrega una columna, que es exactamente lo que
# paso al sumarle el indice de modalidad a cada nodo.
GRAFO_VACIO = trazas_grafo(np.zeros((1, 1)), [''])

_EPOCA = 0
_tarea_pendiente_simular = None
_ultimo_resultado_simulacion = None   # DataFrame de simulate_explicit_overrides, o None
_ultima_seleccion_simulada = None     # (circuito, ventana_i) al que corresponde ese resultado

STATUS = widgets.HTML(
    'Sin simular todavia -- elige variables (opcional) y presiona "Simular".'
)

_knobs_por_id = {k.id: k for k in KNOBS}
# Casillas y no `SelectMultiple`, por el mismo motivo que la lista de vanos: en un
# `SelectMultiple` un clic sin ctrl borra todo lo ya elegido, y aqui justamente se quiere
# simular VARIAS variables a la vez. Cada casilla es independiente y `value` sigue siendo
# la tupla de knob ids, asi que `_reconstruir_controles_knob` no se entera del cambio.
knob_selector_widget = construir_selector_casillas(
    [(k.label, k.id) for k in KNOBS], titulo='', alto='150px', ancho_casilla='230px',
    layout=widgets.Layout(width='100%'),
)
controles_knob_box = widgets.VBox([])
_controles_knob_actuales = {}


def _reconstruir_controles_knob(_change=None):
    global _controles_knob_actuales
    _controles_knob_actuales = {
        knob_id: widget_for_knob(_knobs_por_id[knob_id]) for knob_id in knob_selector_widget.value
    }
    controles_knob_box.children = list(_controles_knob_actuales.values())


knob_selector_widget.observe(_reconstruir_controles_knob, names='value')

boton_simular = widgets.Button(description='Simular', button_style='primary')


_CAPA_VACIA = {'lat': [], 'lon': [], 'hovertext': [], 'customdata': []}


def _redibujar_mapa_predicho(*_ignorado):
    """Repaint puro, CERO llamadas al modelo.

    Antes de la primera simulacion de la seleccion activa el mapa no se dibuja: ni
    tramos, ni equipos, ni leyenda -- solo el aviso de que hay que presionar "Simular".
    Un mapa completo pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma
    forma que un resultado, y esa es justamente la confusion que la fila 2 no puede
    permitirse (D2).

    Con resultado: color de grupo KMeans para lo que el simulador predijo -- la MISMA
    paleta del mapa base, porque es la misma geometria -- y NEGRO para todo lo demas
    (vano sin evento en la ventana, o no seleccionado), igual que la estructura del
    circuito en 01.4.
    """
    circuito, ventana_i, _marcados = _seleccion_actual()
    hay_resultado = (
        _ultimo_resultado_simulacion is not None
        and _ultima_seleccion_simulada == (circuito, ventana_i)
    )
    if not hay_resultado:
        with fig.batch_update():
            for _i in (IDX['pred_clases'] + [IDX['pred_sin_dato'], IDX['pred_marcados'],
                                             IDX['pred_trafos'], IDX['pred_switches']]):
                fig.data[_i].lat, fig.data[_i].lon = [], []
                fig.data[_i].hovertext = []
                fig.data[_i].showlegend = False
            fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = (
                'El mapa simulado aparece al presionar <b>Simular</b>.'
            )
        return

    clases_por_fid = clases_por_fid_para_estado(_ultimo_resultado_simulacion, ESTADO_SIMULADO)
    capas = _capas_de_la_seleccion(clases_por_fid, campo='Criticidad simulada',
                                   nombres_clase=NOMBRES_GRUPOS)
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['pred_clases'][_clase]], capas['clases'][_clase])
            fig.data[IDX['pred_clases'][_clase]].showlegend = True
        # Negro: sin evento en la ventana, o fuera de la seleccion simulada. Un vano que
        # el simulador no puntuo no tiene clase, y la ausencia no es la clase mas baja.
        _volcar_capa(fig.data[IDX['pred_sin_dato']], capas['sin_dato'])
        fig.data[IDX['pred_sin_dato']].name = 'Sin evento / no simulado'
        fig.data[IDX['pred_sin_dato']].line.color = COLOR_SIN_EVENTO
        fig.data[IDX['pred_sin_dato']].showlegend = True
        # 13 queda vacia a proposito: en este mapa lo coloreado ES la seleccion, asi que
        # un halo de "marcado" encima no distingue nada que el color no diga ya.
        _volcar_capa(fig.data[IDX['pred_marcados']], _CAPA_VACIA)
        fig.data[IDX['pred_marcados']].showlegend = False
        fig.data[IDX['pred_trafos']].lat, fig.data[IDX['pred_trafos']].lon = tr['lat'], tr['lon']
        fig.data[IDX['pred_trafos']].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
        fig.data[IDX['pred_switches']].lat, fig.data[IDX['pred_switches']].lon = sw['lat'], sw['lon']
        fig.data[IDX['pred_switches']].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = ''


def _limpiar_resultado_simulacion(_change=None):
    """Circuito o ventana cambiaron: el ultimo resultado ya NO corresponde a la
    seleccion activa -- se descarta (fila 2 vuelve a "Aun no simulado" y el panel de
    importancia se vacia) en vez de mostrar la corrida de OTRA seleccion, que violaria
    la regla anti-confusion (D2)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada, _EPOCA
    _ultimo_resultado_simulacion = None
    _ultima_seleccion_simulada = None
    _EPOCA = siguiente_epoca(_EPOCA)  # invalida cualquier job en vuelo
    _redibujar_mapa_predicho()
    _pintar_grafo(None)
    _pintar_ranking(RANKING_VACIO)


def _pintar_grafo(grafo):
    """Repaint puro del panel del grafo. Un grafo ANULADO no se dibuja a medias: se
    vacian las trazas y se dice por que. `estadistico_colapso` anula cuando las
    compuertas no varian entre vanos -- y su veredicto incluye `effective_rank <= 1`,
    que con menos de 3 vanos se cumple por construccion (la matriz centrada de 1 o 2
    filas tiene rango 1). Dibujar igual seria presentar el grafo experto FIJO como si
    lo hubiera estimado esta seleccion."""
    if grafo is None:
        trazas, mensaje = GRAFO_VACIO, 'Presiona "Simular" para estimar el grafo.'
    elif grafo['voided']:
        trazas = GRAFO_VACIO
        mensaje = (f'Grafo no estimable: las compuertas no varian entre los '
                   f'{grafo["n_vanos"]} vanos de la seleccion.<br>'
                   '<sup>Hacen falta al menos 3 vanos con comportamiento distinto.</sup>')
    else:
        trazas, mensaje = trazas_grafo(grafo['matriz'], FEATURES_MIL), ''

    with fig.batch_update():
        fig.data[IDX['grafo_aristas']].x = trazas['aristas']['x']
        fig.data[IDX['grafo_aristas']].y = trazas['aristas']['y']
        _pesos = trazas['pesos']
        fig.data[IDX['grafo_pesos']].x = _pesos['x']
        fig.data[IDX['grafo_pesos']].y = _pesos['y']
        fig.data[IDX['grafo_pesos']].hovertext = _pesos['hovertext']
        # El tamano codifica el peso relativo DE ESTA seleccion: los pesos absolutos
        # cambian dos ordenes de magnitud entre ventanas y un tamano fijo por valor
        # dejaria el panel vacio o saturado segun cual se mire.
        _maximo = max(_pesos['peso'], default=0.0) or 1.0
        fig.data[IDX['grafo_pesos']].marker.size = [4 + 10 * (p / _maximo) for p in _pesos['peso']]
        fig.data[IDX['grafo_pesos']].marker.color = list(_pesos['peso'])
        # Un nodo por variable, con su NOMBRE al lado y el color de su modo. El rotulo
        # se manda hacia afuera del circulo (a la derecha en la mitad derecha, a la
        # izquierda en la izquierda) para que no se monte sobre las aristas.
        _nodos = trazas['nodos']
        for _i_traza, _modalidad in zip(IDX['grafo_nodos'], MODALIDADES_MIL):
            _cuales = [k for k, col in enumerate(_nodos['indice'])
                       if col in COLUMNAS_MODALIDAD[_modalidad]]
            _traza_nodo = fig.data[_i_traza]
            _traza_nodo.x = [_nodos['x'][k] for k in _cuales]
            _traza_nodo.y = [_nodos['y'][k] for k in _cuales]
            _traza_nodo.text = [_nodos['texto'][k] for k in _cuales]
            _traza_nodo.textposition = ['middle right' if _nodos['x'][k] >= 0
                                        else 'middle left' for k in _cuales]
            _traza_nodo.hovertext = [f'<b>{_nodos["texto"][k]}</b><br>Modo: {_modalidad}'
                                     for k in _cuales]
        fig.layout.annotations[IDX_ANOTACION_GRAFO].text = mensaje


def _simular(epoca_job):
    """Computo pesado -- bloqueante dentro de la corutina (design section A: un job ya
    iniciado no se puede interrumpir). Mapa simulado, grafo e importancia, en ese orden y
    en el mismo job. Guarda y repinta SOLO si `epoca_job` sigue vigente al terminar
    (epoch guard)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
    circuito, ventana_i, marcados = _seleccion_actual()
    seleccion = seleccionar_bolsas(BAG_INDEX, circuito=circuito,
                                   ventana=VENTANAS[ventana_i]['etiqueta'],
                                   marcados=marcados)
    if seleccion['n_bolsas'] == 0:
        aplicar_si_vigente(
            lambda: setattr(STATUS, 'value',
                            'Sin bolsas (vano x ventana) para esta seleccion.'),
            epoca_job=epoca_job, epoca_actual=lambda: _EPOCA,
        )
        return

    overrides = expand_knob_overrides(
        {knob_id: widget.value for knob_id, widget in _controles_knob_actuales.items()}, KNOBS,
    )

    t0 = time.perf_counter()
    resultado, metadata = simular_bolsas(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL, overrides=overrides,
        label_encoders=label_encoders, max_values_imputed=max_values_imputed,
    )
    # El grafo se estima sobre las features OBSERVADAS de la seleccion, no sobre las
    # simuladas: describe a estos vanos, no al escenario hipotetico.
    gates = gates_de_bolsas(MIL, X_INST[seleccion['filas']], seleccion['instance_bag'],
                            seleccion['n_bolsas'])
    grafo = grafo_de_gates(gates, MIL.model.edge_index, n_features=len(FEATURES_MIL))
    ranking = _calcular_ranking(circuito, ventana_i, marcados)
    duracion = time.perf_counter() - t0

    def _escribir():
        global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
        _ultimo_resultado_simulacion = resultado
        _ultima_seleccion_simulada = (circuito, ventana_i)
        grano = f'{len(marcados)} vanos marcados' if marcados else 'todo el circuito'
        cambian = int((resultado['delta_riesgo_ordinal'] != 0).sum())
        avisos = f' | {len(metadata["avisos"])} avisos' if metadata['avisos'] else ''
        STATUS.value = (
            f'{duracion:.2f} s | MIL sobre {metadata["n_vanos"]} bolsas '
            f'({metadata["n_instancias"]:,} eventos) | '
            f'{len(metadata["variables_aplicadas"])} variables aplicadas | '
            f'{cambian} vanos cambian de clase | grafo sobre {grano}'
            f' | importancia ({ranking["n_filas"]:,} muestras){avisos}'
        )
        _redibujar_mapa_predicho()
        _pintar_grafo(grafo)
        _pintar_ranking(ranking)

    aplicar_si_vigente(_escribir, epoca_job=epoca_job, epoca_actual=lambda: _EPOCA)


def _programar_simulacion(*_ignorado):
    global _EPOCA, _tarea_pendiente_simular
    if _tarea_pendiente_simular is not None and not _tarea_pendiente_simular.done():
        _tarea_pendiente_simular.cancel()
    _EPOCA = siguiente_epoca(_EPOCA)
    epoca_job = _EPOCA
    STATUS.value = 'Simulando...'

    async def _tarea():
        try:
            await asyncio.sleep(DEBOUNCE_SEGUNDOS)
        except asyncio.CancelledError:
            return
        _simular(epoca_job)

    _tarea_pendiente_simular = asyncio.ensure_future(_tarea())


boton_simular.on_click(_programar_simulacion)
circuito_widget.observe(_limpiar_resultado_simulacion, names='value')
ventana_widget.observe(_limpiar_resultado_simulacion, names='value')
vano_widget.observe(_redibujar_mapa_predicho, names='value')  # solo redibuja el halo marcado

_redibujar_mapa_predicho()  # primer dibujo: sin simulacion todavia -> "Aun no simulado"
_pintar_grafo(None)


In [ ]:
# --- El panel, ARRIBA y del ancho de la figura (paridad 01.4) -----------------------
# Una sola columna, en el orden en que se usa: circuito -> ventana -> vanos -> variables
# del simulador -> el control de cada variable elegida -> "Simular" -> estado. Cada paso
# depende del anterior, asi que apilarlos evita el zigzag de un flex-wrap donde el boton
# podia quedar antes de los deslizadores que lo alimentan.
#
# El estilo va por CSS y no por `Layout` porque ipywidgets 8 no expone `background`,
# `box-sizing` ni `gap` como traits -- solo `border`, `padding`, `margin` y el flexbox
# basico. `add_class` es la via soportada para lo demas.
ESTILO = widgets.HTML('''
<style>
  .panel-v15 {
    box-sizing: border-box;
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b; font-size: 13px;
  }
  /* Cada grupo, un renglon completo: el panel es una columna, no una grilla. */
  .panel-v15 .grupo-v15 { margin: 0 0 10px 0; width: 100%; }
  .panel-v15 .titulo-v15 { font-weight: 600; margin-bottom: 2px; }
  /* La lista compacta de 01.4: letra 12px, muchas casillas por renglon y scroll propio en
     vez de estirar el panel cuando el circuito tiene cientos de vanos.
     El ancho de cada casilla NO se toca aqui: viaja como estilo inline desde su `Layout`
     y le ganaria a esta hoja igual. */
  .lista-vanos, .lista-variables { font-size: 12px; }
  .lista-vanos .widget-checkbox label,
  .lista-variables .widget-checkbox label { white-space: nowrap; font-weight: 400; }
</style>''')


def _grupo(*hijos):
    """Un bloque del panel: su rotulo y sus controles juntos, como los `div` de 01.4."""
    caja = widgets.VBox(list(hijos), layout=widgets.Layout(align_items='flex-start',
                                                           width='100%'))
    caja.add_class('grupo-v15')
    return caja


def _titulo(texto):
    return widgets.HTML(f'<span class="titulo-v15">{texto}</span>')


vano_widget.caja.add_class('lista-vanos')
knob_selector_widget.caja.add_class('lista-variables')

PANEL = widgets.VBox(
    [
        _grupo(_titulo('Circuito'), circuito_widget),
        _grupo(_titulo('Ventana'), ventana_widget),
        _grupo(vano_widget,
               widgets.HBox([boton_marcar_todos, boton_desmarcar]),
               widgets.HTML('<span style="font-size:12px;color:#5b4a48;">Tambien puedes '
                            'marcar y desmarcar un vano haciendo clic sobre el en el '
                            'mapa.</span>')),
        _grupo(_titulo('Variables del simulador'), knob_selector_widget),
        _grupo(controles_knob_box),
        _grupo(boton_simular),
        _grupo(STATUS),
    ],
    layout=widgets.Layout(
        width=f'{fig.layout.width}px', align_items='flex-start',
        padding='12px 14px', margin='0 0 6px 0',
        border='1px solid #e4c4c0', border_left='4px solid rgb(203,24,29)',
    ),
)
PANEL.add_class('panel-v15')

APP = widgets.VBox([ESTILO, PANEL, fig],
                   layout=widgets.Layout(width=f'{fig.layout.width}px'))
# `display` explicito y una sola vez. `add_class` devuelve el propio widget, asi que
# dejarlo como ultima expresion de la celda hacia que Jupyter lo auto-mostrara ADEMAS del
# display de la celda siguiente: el tablero aparecia dos veces.
display(APP)


In [ ]:
# --- Panel WEB: el mismo tablero, autocontenido y a ancho completo en el navegador ----
# Al ejecutar el cuaderno se escribe un HTML autocontenido y se abre en el navegador, que
# es donde el tablero usa TODO el ancho de la pantalla en vez del de la celda (paridad
# con 01/03/04).
#
# Que SI viaja al navegador: los dos mapas, la nube KMeans con su frontera, la evolucion
# y los dos violines, con circuito, ventana y seleccion de vanos EN VIVO -- todo eso sale
# de datos ya calculados y no necesita el kernel.
#
# Que NO puede viajar: el boton "Simular". El mapa de criticidad simulada, el grafo
# reconstruido y la importancia de variables salen del modelo MIL de torch corriendo en el
# kernel, sobre knobs numericos continuos y combinaciones arbitrarias -- no hay conjunto
# finito que precomputar. Lo que si viaja es la ULTIMA simulacion corrida en el cuaderno,
# como foto congelada, y se borra en cuanto se cambia de circuito o de ventana: mostrar la
# corrida de OTRA seleccion es justo la confusion que separa las dos filas (decision D2).
import json

import plotly.io as pio

DIV_FIGURA = 'simulador-criticidad-vano'

# Los ejes que el JS retoca a mano se resuelven AQUI y no se escriben a mano alla: el
# nombre depende de la posicion del subplot en la grilla, y adivinarlo deja el panel
# reescalando un eje que no es (mismo patron que `_clave_eje` en 01).
def _eje_de(indice_traza, cual):
    ref = getattr(fig.data[indice_traza], f'{cual}axis') or cual
    return f'{cual}axis' + ref[1:]


EJES_WEB = {
    'violinEventosY': _eje_de(IDX['violin_eventos'][0], 'y'),
    'rankingX': _eje_de(IDX['ranking'], 'x'),
}

# Todo lo que sigue arma el HTML autocontenido, y solo corre si se pidio. Va en DOS
# bloques y no en uno: las plantillas del medio son literales, no cuestan nada, y
# dejarlas afuera evita sangrar mil lineas de HTML/JS solo para envolverlas.
if EXPORTAR_PANEL_WEB:
    # Las celdas (vano x ventana) van como COLUMNAS PARALELAS y no como un diccionario por
    # fid: son 111 mil celdas, y una llave de vano repetida por celda pesa mas que todo el
    # resto del contexto junto.
    CELDAS_WEB = {c: [{'fids': [], 'u': [], 'n': [], 'k': []} for _ in VENTANAS]
                  for c in CIRCUITOS}
    for _c, _fid, _vi, _u, _n, _k in zip(
            TABLA['CIRCUITO'].astype(str), TABLA['FID_VANO'].astype(str), TABLA['ventana_i'],
            TABLA['uiti_acumulado'], TABLA['num_eventos'], CLASE_TABLA):
        _celda = CELDAS_WEB[str(_c)][int(_vi)]
        _celda['fids'].append(str(_fid))
        _celda['u'].append(round(float(_u), 3))
        _celda['n'].append(int(_n))
        _celda['k'].append(int(_k))

    # La foto de la ultima simulacion: solo su SELECCION viaja. Las trazas del mapa simulado,
    # el grafo y el ranking ya estan dibujadas en la figura que se serializa, asi que el JS no
    # necesita los datos -- solo saber a que circuito y ventana pertenecen para borrarlos en
    # cuanto la seleccion se mueva.
    SIMULADO_WEB = (
        None if _ultima_seleccion_simulada is None
        else {'circuito': _ultima_seleccion_simulada[0], 'ventana': int(_ultima_seleccion_simulada[1])}
    )

    # --- Payload del simulador: pesos, instancias y catalogo de controles ---------------
    # `extraer_pesos_mil` deja afuera decoders, clasificadores, regresores y reliability
    # heads: bajo fusion='film' no participan de `p_bag` y serian 61.268 parametros de
    # payload que no dicen nada. `tests/test_mil_web_export.py` clava el forward de
    # referencia contra el modulo de torch, y el JS del panel es su transcripcion.
    from chec_local_interpreter.mil_web_export import (
        a_base64,
        extraer_pesos_mil,
        pesos_a_json,
    )
    from chec_local_interpreter.simulator import _coerce_original_value_for_model

    PESOS_WEB = pesos_a_json(extraer_pesos_mil(MIL))

    _circuitos_sim = list(CIRCUITOS_SIMULABLES or [circuito_widget.value])
    _claves = BAG_INDEX.keys
    _col_circ = _claves['CIRCUITO'].astype(str).to_numpy()
    _col_fid = _claves['FID_VANO'].astype(str).to_numpy()
    _col_vent = _claves['VENTANA'].astype(str).to_numpy()
    _offsets = np.asarray(BAG_INDEX.offsets, dtype=np.int64)
    _conteos = np.asarray(BAG_INDEX.counts, dtype=np.int64)
    _i_por_etiqueta = {v['etiqueta']: int(v['i']) for v in VENTANAS}

    SIMULABLES_WEB = {}
    for _c in _circuitos_sim:
        _bolsas = np.flatnonzero(_col_circ == str(_c))
        if _bolsas.size == 0:
            continue
        # Las instancias del circuito, contiguas y en el orden de sus bolsas: asi el JS
        # arma una seleccion copiando rangos, sin indexar fila por fila.
        _filas = np.concatenate([np.arange(_offsets[b], _offsets[b] + _conteos[b])
                                 for b in _bolsas])
        _X_circ = np.asarray(X_INST, dtype=np.float32)[_filas]
        _cnt = _conteos[_bolsas]
        _inicio = np.concatenate([[0], np.cumsum(_cnt)[:-1]])
        SIMULABLES_WEB[str(_c)] = {
            'X': {'b64': a_base64(_X_circ), 'forma': list(_X_circ.shape), 'tipo': 'float32'},
            'fid': [_col_fid[b] for b in _bolsas],
            'ventana': [_i_por_etiqueta.get(_col_vent[b], -1) for b in _bolsas],
            'inicio': [int(v) for v in _inicio],
            'conteo': [int(v) for v in _cnt],
        }

    # El catalogo de controles. Para un knob CATEGORICO el valor que ve el modelo se resuelve
    # AQUI con el mismo `_coerce_original_value_for_model` que usa el cuaderno -- el navegador
    # no lleva los LabelEncoder, y recodificar a mano alla seria una segunda verdad.
    _posicion_feature = {str(nombre): i for i, nombre in enumerate(FEATURES_MIL)}
    KNOBS_WEB = []
    for _k in KNOBS:
        if _k.kind == 'constant':
            continue
        _indices = [_posicion_feature[f] for f in _k.feature_names if f in _posicion_feature]
        if not _indices:
            continue
        _entrada = {'id': _k.id, 'label': _k.label, 'kind': _k.kind, 'indices': _indices,
                    'bounds': [float(v) for v in _k.bounds] if _k.bounds else None,
                    'step': float(_k.step) if _k.step else None}
        if _k.kind == 'categorical':
            _opciones = []
            for _cat in (_k.categories or ()):
                try:
                    _valor = _coerce_original_value_for_model(
                        _k.feature_names[0], _cat, label_encoders=label_encoders,
                        max_values_imputed=max_values_imputed)
                except (ValueError, TypeError):
                    continue
                _opciones.append({'etiqueta': str(_cat), 'valor': float(_valor)})
            if not _opciones:
                continue
            _entrada['opciones'] = _opciones
        KNOBS_WEB.append(_entrada)

    assert any(_k['kind'] == 'numeric' for _k in KNOBS_WEB), (
        'sin knobs numericos no hay barrido de relevancia que dibujar')
    _mb_pesos = len(json.dumps(PESOS_WEB)) / 1024 ** 2
    _mb_inst = sum(len(_v['X']['b64']) for _v in SIMULABLES_WEB.values()) / 1024 ** 2
    print(f'simulador web: {len(KNOBS_WEB)} controles | pesos {_mb_pesos:,.2f} MB | '
          f'instancias de {len(SIMULABLES_WEB)} circuito(s) {_mb_inst:,.2f} MB '
          f'({", ".join(SIMULABLES_WEB)})')

    CONTEXTO_WEB = {
        'div': DIV_FIGURA,
        'circuitos': CIRCUITOS,
        'ventanas': [{'i': int(v['i']), 'etiqueta': v['etiqueta'], 'periodo': v['periodo']}
                     for v in VENTANAS],
        'geo': GEO_POR_CIRCUITO,
        'trafos': TRAFOS,
        'switches': SWITCHES,
        'celdas': CELDAS_WEB,
        'vanos': VANOS_POR_CIRCUITO,
        'nombresGrupos': NOMBRES_GRUPOS,
        'coloresGrupos': COLORES_GRUPOS,
        'coloresVanos': COLORES_VANOS,
        'nCupos': N_CUPOS_EVOLUCION,
        'idx': IDX,
        'ejes': EJES_WEB,
        'idxAnotSimulado': IDX_ANOTACION_SIMULADO,
        'idxAnotGrafo': IDX_ANOTACION_GRAFO,
        'marcaVano': MARCA_VANO,
        'pasoVertice': PASO_VERTICE,
        'maxCortesTramo': MAX_CORTES_TRAMO,
        'circuitoInicial': circuito_widget.value,
        'ventanaInicial': int(ventana_widget.value),
        'vanosIniciales': list(vano_widget.value),
        'simulado': SIMULADO_WEB,
        # --- simulador ---
        'pesos': PESOS_WEB,
        'simulables': SIMULABLES_WEB,
        'knobs': KNOBS_WEB,
        'nFeatures': len(FEATURES_MIL),
        'featuresMil': [str(f) for f in FEATURES_MIL],
        'modalidades': MODALIDADES_MIL,
        'coloresModalidad': [COLORES_MODALIDAD[_m] for _m in MODALIDADES_MIL],
        'columnasModalidad': [sorted(COLUMNAS_MODALIDAD[_m]) for _m in MODALIDADES_MIL],
    }
    assert set(CONTEXTO_WEB['celdas']) == set(CIRCUITOS)
    assert set(CONTEXTO_WEB['simulables']).issubset(set(CIRCUITOS))
    assert all(len(_s['fid']) == len(_s['ventana']) == len(_s['inicio']) == len(_s['conteo'])
               for _s in SIMULABLES_WEB.values()), 'las columnas de bolsas van alineadas'
    assert all(_s['X']['forma'][1] == len(FEATURES_MIL) for _s in SIMULABLES_WEB.values()), (
        'la matriz de instancias tiene que traer TODAS las columnas del modelo')
    assert CONTEXTO_WEB['circuitoInicial'] in CONTEXTO_WEB['circuitos']
    assert all(len(_v) == len(VENTANAS) for _v in CELDAS_WEB.values()), (
        'cada circuito lleva una entrada por ventana, aunque quede vacia')
    assert sum(len(_v['fids']) for _c in CELDAS_WEB.values() for _v in _c) == len(TABLA), (
        'ninguna celda de TABLA se puede perder al pasar a columnas paralelas')

_opciones_circuito_web = ''.join(
    f'<option value="{_c}"{" selected" if _c == circuito_widget.value else ""}>{_c}</option>'
    for _c in CIRCUITOS)
_leyenda_grupos_web = ''.join(
    f'<span><span style="display:inline-block;width:34px;height:14px;background:{_color};'
    f'vertical-align:middle;margin-right:8px;"></span>{_nombre}</span>'
    for _nombre, _color in zip(NOMBRES_GRUPOS, COLORES_GRUPOS))

# Mismo tema rojo del panel de widgets y de 01/03/04: un color de chrome significa lo
# mismo al pasar de un tablero a otro.
PANEL_HTML_WEB = f'''
<style>
  .panel-sim {{
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif; font-size: 15px;
    display: flex; flex-wrap: wrap; gap: 12px 18px; align-items: flex-end;
    max-width: 100%; margin: 0 0 8px 0; padding: 12px 14px; box-sizing: border-box;
    border: 1px solid #e4c4c0; border-left: 4px solid rgb(203,24,29);
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b;
  }}
  .panel-sim label {{ display: block; font-weight: 600; margin-bottom: 6px; }}
  .panel-sim select {{
    font: inherit; padding: 6px 10px; border: 1px solid #c9a9a5; border-radius: 4px;
    background: #fff; color: #2b2b2b; min-width: 220px;
  }}
  .panel-sim button {{
    font: inherit; padding: 6px 12px; border: 1px solid #c9a9a5; border-radius: 4px;
    background: #fff; color: #2b2b2b; cursor: pointer;
  }}
  .panel-sim button:hover {{ background: #f6e9e7; }}
  .panel-sim input[type="range"] {{ height: 22px; accent-color: rgb(203,24,29); }}
  /* La lista de vanos con scroll propio: un circuito de 1.377 vanos no puede estirar el
     panel hasta empujar la figura fuera de la pantalla. */
  #sim-vanos {{
    display: flex; flex-wrap: wrap; gap: 2px 14px; max-height: 130px; overflow-y: auto;
    width: 100%; padding: 8px 10px; box-sizing: border-box; font-size: 13px;
    border: 1px solid #e4c4c0; border-radius: 4px; background: #fff;
  }}
  #sim-vanos label {{ display: inline-flex; align-items: center; gap: 4px;
                      font-weight: 400; margin: 0; white-space: nowrap; }}
  #sim-knobs {{
    display: flex; flex-wrap: wrap; gap: 2px 14px; max-height: 96px; overflow-y: auto;
    width: 100%; padding: 8px 10px; box-sizing: border-box; font-size: 13px;
    border: 1px solid #e4c4c0; border-radius: 4px; background: #fff;
  }}
  #sim-knobs label {{ display: inline-flex; align-items: center; gap: 4px;
                     font-weight: 400; margin: 0; white-space: nowrap; }}
  .sim-knob {{ display: flex; align-items: center; gap: 10px; margin: 6px 0;
              font-size: 13px; }}
  .sim-knob > span:first-child {{ min-width: 260px; font-weight: 600; }}
  .sim-knob input[type="range"] {{ flex: 1; accent-color: rgb(203,24,29); }}
  .sim-knob .sim-valor {{ min-width: 110px; text-align: right; font-variant-numeric:
                         tabular-nums; }}
  .sim-fila {{ flex-basis: 100%; display: flex; flex-wrap: wrap; gap: 8px 22px;
               align-items: center; }}
  .sim-nota {{ flex-basis: 100%; font-size: 13px; color: #7a5c58; margin: 0; }}
</style>
<div class="panel-sim">
  <div><label for="sim-circuito">Circuito</label>
       <select id="sim-circuito">{_opciones_circuito_web}</select></div>
  <div style="flex:1; min-width:320px;">
    <label for="sim-ventana">Ventana</label>
    <div style="display:flex; align-items:center; gap:10px;">
      <input type="range" id="sim-ventana" min="0" max="{len(VENTANAS) - 1}"
             value="{int(ventana_widget.value)}" step="1" style="flex:1;">
      <span id="sim-ventana-txt" style="font-weight:600; white-space:nowrap;"></span>
    </div>
  </div>
  <div class="sim-fila">
    <span style="font-weight:600;">Vanos marcados</span>
    <button id="sim-marcar-todos" type="button">Marcar todos</button>
    <button id="sim-desmarcar" type="button">Desmarcar</button>
    <span id="sim-conteo" style="color:#5b4a48;"></span>
  </div>
  <div id="sim-vanos"></div>
  <p class="sim-nota">Tambien puedes marcar y desmarcar un vano haciendo clic sobre el en
     el mapa de arriba.</p>
  <div class="sim-fila" style="font-size:13px; color:#5b4a48;">
    <span style="font-weight:600;">Criticidad (grupos KMeans de 01.4):</span>
    {_leyenda_grupos_web}
    <span><span style="display:inline-block;width:34px;height:14px;
      background:{COLOR_SIN_EVENTO};vertical-align:middle;margin-right:8px;"></span>Sin
      evento en la ventana</span>
    <span><span style="display:inline-block;width:14px;height:14px;background:{COLOR_TRAFO};
      border-radius:50%;margin-right:8px;"></span>Transformador</span>
    <span><span style="display:inline-block;width:14px;height:14px;background:{COLOR_SWITCH};
      border-radius:50%;margin-right:8px;"></span>Interruptor</span>
  </div>
  <p class="sim-nota" id="sim-aviso"></p>
  <div class="sim-fila" style="border-top:1px solid #e4c4c0; padding-top:10px;">
    <span style="font-weight:600;">Simulador</span>
    <button id="sim-correr" type="button"
            style="background:rgb(203,24,29); color:#fff; border-color:rgb(203,24,29);
                   font-weight:600;">Simular</button>
    <span id="sim-estado" style="color:#5b4a48;"></span>
  </div>
  <div class="sim-fila">
    <span style="font-size:13px; color:#5b4a48;">Variables a intervenir (sin ninguna, el
      escenario simulado es el observado):</span>
  </div>
  <div id="sim-knobs"></div>
  <div id="sim-controles" style="flex-basis:100%;"></div>
</div>
'''

# El JS se arma con un marcador de texto y NO con f-string ni %: el JSON del contexto trae
# llaves y el CSS del JS trae "%", y escapar todo eso a mano es donde se cuelan los
# errores (mismo patron que 01/03/04).
PANEL_JS_WEB_TEMPLATE = r'''
<script type="text/javascript">
// --- El forward del MIL, en el navegador ---------------------------------------------
// Transcripcion directa de `chec_local_interpreter.mil_web_export.predecir_numpy`, que a
// su vez esta clavada contra el modulo de torch en `tests/test_mil_web_export.py`. Cada
// funcion de aqui tiene su gemela alla; si una cambia, la otra tiene que cambiar con ella.
// Es lo unico que impide que el boton "Simular" del HTML conteste con seguridad una
// criticidad equivocada.
var MIL = (function () {
  'use strict';

  // Una matriz es {d: Float32Array plano en orden de filas, m: filas, n: columnas}.
  function mat(m, n) { return {d: new Float32Array(m * n), m: m, n: n}; }

  function desB64(spec) {
    var binario = atob(spec.b64);
    var bytes = new Uint8Array(binario.length);
    for (var i = 0; i < binario.length; i++) { bytes[i] = binario.charCodeAt(i); }
    var vista = spec.tipo === 'int32' ? new Int32Array(bytes.buffer)
                                      : new Float32Array(bytes.buffer);
    var forma = spec.forma;
    return {d: vista, m: forma.length > 1 ? forma[0] : 1,
            n: forma.length > 1 ? forma[1] : (forma[0] || 0)};
  }

  // Convierte, en su lugar, todo `{b64, forma, tipo}` del arbol de pesos en una matriz.
  function hidratar(nodo) {
    if (nodo === null || typeof nodo !== 'object') { return nodo; }
    if (Object.prototype.hasOwnProperty.call(nodo, 'b64')) { return desB64(nodo); }
    if (Array.isArray(nodo)) { return nodo.map(hidratar); }
    var salida = {};
    for (var clave in nodo) {
      if (Object.prototype.hasOwnProperty.call(nodo, clave)) {
        salida[clave] = hidratar(nodo[clave]);
      }
    }
    return salida;
  }

  // `A (m x k) @ B (k x n) + b`. Orden i-k-j a proposito: recorre B por filas
  // contiguas, que es entre 3 y 5 veces mas rapido que i-j-k sobre typed arrays y es
  // lo que hace viable el barrido de relevancia sin congelar la pestania.
  function matmul(A, B, sesgo) {
    var m = A.m, k = A.n, n = B.n;
    var C = mat(m, n), a = A.d, b = B.d, c = C.d, i, j, p, base, fila, valor;
    if (sesgo) {
      for (i = 0; i < m; i++) {
        base = i * n;
        for (j = 0; j < n; j++) { c[base + j] = sesgo.d[j]; }
      }
    }
    for (i = 0; i < m; i++) {
      base = i * n;
      fila = i * k;
      for (p = 0; p < k; p++) {
        valor = a[fila + p];
        if (valor === 0) { continue; }
        var desplazamiento = p * n;
        for (j = 0; j < n; j++) { c[base + j] += valor * b[desplazamiento + j]; }
      }
    }
    return C;
  }

  function relu(A) {
    for (var i = 0; i < A.d.length; i++) { if (A.d[i] < 0) { A.d[i] = 0; } }
    return A;
  }

  // LayerNorm sobre la ULTIMA dimension, con varianza poblacional (sin correccion de
  // Bessel), que es la que usa torch.
  function layernorm(A, w, b, eps) {
    var m = A.m, n = A.n, d = A.d, i, j, base, suma, media, acumulado, inv;
    for (i = 0; i < m; i++) {
      base = i * n;
      suma = 0;
      for (j = 0; j < n; j++) { suma += d[base + j]; }
      media = suma / n;
      acumulado = 0;
      for (j = 0; j < n; j++) { var dif = d[base + j] - media; acumulado += dif * dif; }
      inv = 1 / Math.sqrt(acumulado / n + eps);
      for (j = 0; j < n; j++) {
        d[base + j] = (d[base + j] - media) * inv * w.d[j] + b.d[j];
      }
    }
    return A;
  }

  function softmaxFilas(A) {
    var m = A.m, n = A.n, d = A.d, i, j, base, maximo, suma;
    for (i = 0; i < m; i++) {
      base = i * n;
      maximo = -Infinity;
      for (j = 0; j < n; j++) { if (d[base + j] > maximo) { maximo = d[base + j]; } }
      suma = 0;
      for (j = 0; j < n; j++) { d[base + j] = Math.exp(d[base + j] - maximo); suma += d[base + j]; }
      for (j = 0; j < n; j++) { d[base + j] /= suma; }
    }
    return A;
  }

  function aplicarCapas(x, capas) {
    for (var i = 0; i < capas.length; i++) {
      var capa = capas[i];
      if (capa.tipo === 'linear') { x = matmul(x, capa.W, capa.b); }
      else if (capa.tipo === 'relu') { x = relu(x); }
      else if (capa.tipo === 'layernorm') { x = layernorm(x, capa.w, capa.b, capa.eps); }
      else if (capa.tipo === 'dropout') { /* identidad en eval */ }
      else { throw new Error('capa desconocida: ' + capa.tipo); }
    }
    return x;
  }

  function columnas(X, indices) {
    var m = X.m, n = indices.d.length, salida = mat(m, n), i, j;
    for (i = 0; i < m; i++) {
      for (j = 0; j < n; j++) { salida.d[i * n + j] = X.d[i * X.n + indices.d[j]]; }
    }
    return salida;
  }

  function concatenar(partes) {
    var m = partes[0].m, n = 0, i, k, j, desplazamiento = 0;
    for (i = 0; i < partes.length; i++) { n += partes[i].n; }
    var salida = mat(m, n);
    for (k = 0; k < partes.length; k++) {
      var parte = partes[k];
      for (i = 0; i < m; i++) {
        for (j = 0; j < parte.n; j++) {
          salida.d[i * n + desplazamiento + j] = parte.d[i * parte.n + j];
        }
      }
      desplazamiento += parte.n;
    }
    return salida;
  }

  function codificar(pesos, X) {
    var salidas = [], k;
    for (k = 0; k < pesos.modalidades.length; k++) {
      var modalidad = pesos.modalidades[k];
      var entrada = columnas(X, modalidad.indices);
      var atencion = modalidad.atencion;
      // La puerta de atencion por feature normaliza una COPIA: la entrada cruda es la
      // que se reescala despues (`x * softmax(scorer(norm(x))) * escala`).
      var normalizada = layernorm({d: entrada.d.slice(), m: entrada.m, n: entrada.n},
                                  atencion.norma.w, atencion.norma.b, atencion.norma.eps);
      var pesosAtencion = softmaxFilas(
        matmul(normalizada, atencion.scorer.W, atencion.scorer.b));
      var escalada = mat(entrada.m, entrada.n);
      for (var i = 0; i < escalada.d.length; i++) {
        escalada.d[i] = entrada.d[i] * pesosAtencion.d[i] * atencion.escala;
      }
      salidas.push(aplicarCapas(escalada, modalidad.red));
    }
    return concatenar(salidas);
  }

  // Pooling por atencion sobre segmentos. La resta del maximo por bolsa es el mismo
  // apanio de estabilidad numerica que hace torch y no cambia el resultado.
  function agrupar(pesos, z, instanceBag, nBags) {
    var proyectado = matmul(z, pesos.pool.proyeccion.W, pesos.pool.proyeccion.b);
    for (var i = 0; i < proyectado.d.length; i++) {
      proyectado.d[i] = Math.tanh(proyectado.d[i]);
    }
    var puntajes = matmul(proyectado, pesos.pool.cabeza.W, pesos.pool.cabeza.b);
    var nInst = z.m, b;
    var maximos = new Float64Array(nBags).fill(-Infinity);
    for (i = 0; i < nInst; i++) {
      b = instanceBag[i];
      if (puntajes.d[i] > maximos[b]) { maximos[b] = puntajes.d[i]; }
    }
    var exp = new Float64Array(nInst), suma = new Float64Array(nBags);
    for (i = 0; i < nInst; i++) {
      b = instanceBag[i];
      exp[i] = Math.exp(puntajes.d[i] - maximos[b]);
      suma[b] += exp[i];
    }
    var zBolsa = mat(nBags, z.n);
    for (i = 0; i < nInst; i++) {
      b = instanceBag[i];
      var a = exp[i] / Math.max(suma[b], 1e-12);
      for (var j = 0; j < z.n; j++) { zBolsa.d[b * z.n + j] += a * z.d[i * z.n + j]; }
    }
    return zBolsa;
  }

  /** u-hat por bolsa y las compuertas por arista. Espejo de `predecir_numpy`. */
  function predecir(pesos, X, instanceBag, nBags) {
    if (X.m === 0) { return {u: new Float64Array(0), compuertas: mat(0, 0)}; }
    var z1 = codificar(pesos, X);
    var zBolsa = agrupar(pesos, z1, instanceBag, nBags);
    var compuertas = matmul(zBolsa, pesos.compuertas.W, pesos.compuertas.b);
    for (var i = 0; i < compuertas.d.length; i++) {
      compuertas.d[i] = 2 / (1 + Math.exp(-compuertas.d[i]));
    }

    // Propagacion por arista: acumula en la COLUMNA destino. Una columna que no
    // aparezca en `aristaColumnas` queda intacta, igual que el `index_add(1, ...)`.
    var nAristas = pesos.aristaValores.d.length;
    var propagado = {d: X.d.slice(), m: X.m, n: X.n};
    var filas = pesos.aristaFilas.d, cols = pesos.aristaColumnas.d;
    var valores = pesos.aristaValores.d, alpha = pesos.alpha;
    for (i = 0; i < X.m; i++) {
      var bolsa = instanceBag[i], baseX = i * X.n, baseG = bolsa * nAristas;
      for (var e = 0; e < nAristas; e++) {
        propagado.d[baseX + cols[e]] +=
          alpha * compuertas.d[baseG + e] * valores[e] * X.d[baseX + filas[e]];
      }
    }

    var z2 = codificar(pesos, propagado);
    var zBolsa2 = agrupar(pesos, z2, instanceBag, nBags);

    var embed = pesos.embedDim, nMod = pesos.nModalidades;
    var indice = pesos.film.indiceModulada;
    var modulada = mat(nBags, embed), contexto = mat(nBags, (nMod - 1) * embed);
    for (i = 0; i < nBags; i++) {
      var destino = 0;
      for (var k = 0; k < nMod; k++) {
        for (var j = 0; j < embed; j++) {
          var v = zBolsa2.d[i * zBolsa2.n + k * embed + j];
          if (k === indice) { modulada.d[i * embed + j] = v; }
          else { contexto.d[i * contexto.n + destino + j] = v; }
        }
        if (k !== indice) { destino += embed; }
      }
    }
    var gamma = matmul(contexto, pesos.film.gamma.W, pesos.film.gamma.b);
    var beta = matmul(contexto, pesos.film.beta.W, pesos.film.beta.b);
    var zFilm = mat(nBags, embed);
    for (i = 0; i < zFilm.d.length; i++) {
      zFilm.d[i] = modulada.d[i] * (1 + gamma.d[i]) + beta.d[i];
    }
    var pBag = matmul(zFilm, pesos.film.cabeza.W, pesos.film.cabeza.b);
    var u = new Float64Array(nBags);
    for (i = 0; i < nBags; i++) { u[i] = Math.expm1(pBag.d[i]); }
    return {u: u, compuertas: compuertas};
  }

  /** Clase por centroide mas cercano; espejo de `asignar_clase`. */
  function clase(nObs, u, geometria, eps) {
    eps = eps === undefined ? 1e-6 : eps;
    var offset = geometria.offset.d, escala = geometria.scale.d;
    var centroides = geometria.centroides, nC = centroides.m;
    var salida = new Int32Array(u.length), i, c;
    for (i = 0; i < u.length; i++) {
      var x0 = geometria.logs[0] ? Math.log10(Math.max(nObs[i], eps)) : nObs[i];
      var x1 = geometria.logs[1] ? Math.log10(Math.max(u[i], eps)) : u[i];
      var z0 = (x0 - offset[0]) / escala[0], z1 = (x1 - offset[1]) / escala[1];
      var mejor = 0, mejorD = Infinity;
      for (c = 0; c < nC; c++) {
        var d0 = z0 - centroides.d[c * 2], d1 = z1 - centroides.d[c * 2 + 1];
        var d2 = d0 * d0 + d1 * d1;
        if (d2 < mejorD) { mejorD = d2; mejor = c; }
      }
      salida[i] = mejor;
    }
    return salida;
  }

  /** `estadistico_colapso`'s effective rank, sin SVD: `||C||_F^4 / ||C^T C||_F^2`. */
  function rangoEfectivo(compuertas) {
    var m = compuertas.m, n = compuertas.n, d = compuertas.d, i, j, k;
    if (m === 0) { return 0; }
    var centrada = new Float64Array(m * n);
    for (j = 0; j < n; j++) {
      var suma = 0;
      for (i = 0; i < m; i++) { suma += d[i * n + j]; }
      var media = suma / m;
      for (i = 0; i < m; i++) { centrada[i * n + j] = d[i * n + j] - media; }
    }
    var energia = 0;
    for (i = 0; i < centrada.length; i++) { energia += centrada[i] * centrada[i]; }
    if (energia <= 0) { return 0; }
    var normaGram = 0;
    for (j = 0; j < n; j++) {
      for (k = 0; k < n; k++) {
        var g = 0;
        for (i = 0; i < m; i++) { g += centrada[i * n + j] * centrada[i * n + k]; }
        normaGram += g * g;
      }
    }
    return energia * energia / normaGram;
  }

  return {mat: mat, hidratar: hidratar, predecir: predecir, clase: clase,
          rangoEfectivo: rangoEfectivo, desB64: desB64};
})();


(function () {
  var CTX = __CTX_JSON__;
  var d = document;
  var CIRC = CTX.circuitoInicial;
  var MARCADOS = {};           // fid -> true. Objeto y no Set: mismo estado que las casillas.
  var RECENTRADO = null;       // ultimo circuito encuadrado; el encuadre NO se rehace por ventana
  var ULTIMA_VISTA = null;     // el encuadre que puso ESTE panel, para no pisar al usuario
  // A que seleccion corresponde lo que muestran la fila 2, el grafo y el ranking. Nace
  // con la foto que trajo el cuaderno y la reemplaza cada simulacion hecha aqui.
  var SIM_VIGENTE = CTX.simulado
    ? {circuito: CTX.simulado.circuito, ventana: CTX.simulado.ventana} : null;
  var MARGEN_ENCUADRE = 0.9;   // deja borde: el circuito no tiene que tocar los limites
  var TESELA_PX = 512;         // MapLibre proyecta con teselas de 512 px, no de 256
  var GEO_DENSO = {};          // cache de densificacion por circuito

  CTX.vanosIniciales.forEach(function (f) { MARCADOS[String(f)] = true; });

  function geoDe(circ) { return CTX.geo[circ] || {fids: [], lat: [], lon: [], bounds: null}; }
  function ventanaActual() {
    var el = d.getElementById('sim-ventana');
    var v = el ? (parseInt(el.value, 10) || 0) : 0;
    return Math.max(0, Math.min(CTX.ventanas.length - 1, v));
  }
  function marcadosLista() { return Object.keys(MARCADOS); }

  // --- Vertices para el hover del vano (paridad 01) ---------------------------------
  // El hover de una traza de lineas en Scattermap NO se resuelve contra la linea sino
  // contra sus VERTICES: plotly.js mide la distancia del cursor a cada punto y descarta
  // lo que quede a mas de `hoverdistance`, asi que el ancho de la linea no participa del
  // calculo. Los tramos de MVLINSEC.shp traen EXACTAMENTE 2 vertices (60.053 de 60.053
  // medidos), uno en cada extremo, y en un vano largo el centro quedaba a mas de 30 px de
  // los dos y no mostraba ninguna etiqueta. Se interpolan vertices cada ~25 m.
  // Corre en el NAVEGADOR y solo sobre el circuito activo: medido, densificar del lado de
  // Python llevaria el peor circuito de 4.131 a 22.371 puntos y ~2,8 MB de etiquetas por
  // capa, que es justo lo que no puede viajar.
  function densificar(la, lo) {
    if (!la || la.length < 2) { return [la || [], lo || []]; }
    var oLa = [la[0]], oLo = [lo[0]], i, j, n, dLa, dLo;
    for (i = 1; i < la.length; i++) {
      dLa = la[i] - la[i - 1];
      dLo = lo[i] - lo[i - 1];
      n = Math.ceil(Math.max(Math.abs(dLa), Math.abs(dLo)) / CTX.pasoVertice);
      n = Math.max(1, Math.min(CTX.maxCortesTramo, n));
      for (j = 1; j < n; j++) {
        oLa.push(la[i - 1] + dLa * j / n);
        oLo.push(lo[i - 1] + dLo * j / n);
      }
      oLa.push(la[i]);
      oLo.push(lo[i]);
    }
    return [oLa, oLo];
  }

  function geoDenso(circ) {
    if (GEO_DENSO[circ]) { return GEO_DENSO[circ]; }
    var geo = geoDe(circ), la = [], lo = [], i, den;
    for (i = 0; i < geo.fids.length; i++) {
      den = densificar(geo.lat[i], geo.lon[i]);
      la.push(den[0]);
      lo.push(den[1]);
    }
    GEO_DENSO[circ] = {lat: la, lon: lo};
    return GEO_DENSO[circ];
  }

  function capaVacia() { return {lat: [], lon: [], txt: [], cd: []}; }

  // --- Encuadre del circuito ---------------------------------------------------------
  // Un grado de latitud y uno de longitud NO ocupan los mismos pixeles, y menos en un
  // viewport apaisado. La formula vieja derivaba el zoom del span mayor en GRADOS y no
  // miraba el tamanio del mapa: medido en Chrome con el mapa en 1553 x 328 px, DON23L13
  // quedaba en el 21% del ancho y el 119% del ALTO -- centrado pero recortado arriba y
  // abajo, que es lo que se lee como "no se fue al circuito nuevo". Esto es un fitBounds
  // de verdad: proyecta el bounding box y toma la dimension que se queda sin lugar
  // primero.
  function mercatorY(lat) {
    var r = lat * Math.PI / 180;
    return (1 - Math.log(Math.tan(r) + 1 / Math.cos(r)) / Math.PI) / 2;
  }

  // El tamanio REAL del canvas de MapLibre. Antes de que monte no existe, asi que se cae
  // al dominio del subplot -- y los repintados diferidos vuelven a encuadrar con la
  // medida buena.
  function pxDelMapa(gd) {
    var sub = gd._fullLayout.map && gd._fullLayout.map._subplot;
    var cv = sub && sub.map && sub.map.getCanvas && sub.map.getCanvas();
    if (cv && cv.clientWidth > 0 && cv.clientHeight > 0) {
      return [cv.clientWidth, cv.clientHeight];
    }
    var dom = gd._fullLayout.map.domain;
    return [gd._fullLayout.width * (dom.x[1] - dom.x[0]),
            gd._fullLayout.height * (dom.y[1] - dom.y[0])];
  }

  function encuadrar(gd) {
    var b = geoDe(CIRC).bounds;
    if (!b) { return; }
    var px = pxDelMapa(gd);
    var fx = Math.max(Math.abs(b[3] - b[2]) / 360, 1e-12);
    var fy = Math.max(Math.abs(mercatorY(b[0]) - mercatorY(b[1])), 1e-12);
    var escala = Math.min(px[0] * MARGEN_ENCUADRE / (TESELA_PX * fx),
                          px[1] * MARGEN_ENCUADRE / (TESELA_PX * fy));
    // Techo para el circuito de un solo vano; piso bajo porque en un viewport bajo y
    // ancho encuadrar un circuito alto puede pedir menos de 9, y recortarlo era el bug.
    var vista = {center: {lat: (b[0] + b[1]) / 2, lon: (b[2] + b[3]) / 2},
                 zoom: Math.min(15, Math.max(3, Math.log(escala) / Math.LN2))};
    // Los dos mapas comparten encuadre a proposito: la comparacion fila 1 contra fila 2
    // solo se sostiene si miran exactamente la misma geografia.
    Plotly.relayout(gd, {'map.center': vista.center, 'map.zoom': vista.zoom,
                         'map2.center': vista.center, 'map2.zoom': vista.zoom});
    ULTIMA_VISTA = vista;
  }

  // Si el usuario movio el mapa a mano, su vista manda: reencuadrar al cambiar el tamanio
  // de la ventana se la borraria sin que el hubiera pedido nada.
  function camaraSinTocar(gd) {
    if (!ULTIMA_VISTA) { return true; }
    var c = gd._fullLayout.map.center, z = gd._fullLayout.map.zoom;
    return Math.abs(c.lat - ULTIMA_VISTA.center.lat) < 1e-6 &&
           Math.abs(c.lon - ULTIMA_VISTA.center.lon) < 1e-6 &&
           Math.abs(z - ULTIMA_VISTA.zoom) < 1e-6;
  }

  // Un vano dentro de una capa: la polilinea densificada, su separador, y despues el
  // guion horizontal de cada extremo. El guion no puede ser un marcador -- `marker.symbol`
  // de Scattermap solo acepta iconos del sprite del estilo del mapa y ahi no hay ninguna
  // linea horizontal -- asi que va como dos segmentos mas DENTRO de la misma capa, con lo
  // que hereda su color y su ancho sin agregar ninguna traza.
  // `push.apply` y NO `concat`: concat devuelve un array NUEVO, asi que acumular vano a
  // vano sobre una capa de 22.371 puntos la copia entera 1.377 veces -- cuadratico, y se
  // siente al mover el deslizador de ventana. Los trozos son de a lo sumo
  // `maxCortesTramo` elementos, muy por debajo del tope de argumentos de `apply`.
  function agregarVano(capa, fid, dLa, dLo, oLa, oLo, etiqueta) {
    var k, ex, ie;
    capa.lat.push.apply(capa.lat, dLa); capa.lat.push(null);
    capa.lon.push.apply(capa.lon, dLo); capa.lon.push(null);
    for (k = 0; k < dLa.length; k++) { capa.txt.push(etiqueta); capa.cd.push(fid); }
    capa.txt.push(''); capa.cd.push(fid);
    for (ex = 0; ex < 2; ex++) {
      ie = ex === 0 ? 0 : oLa.length - 1;
      capa.lat.push(oLa[ie], oLa[ie], null);
      capa.lon.push(oLo[ie] - CTX.marcaVano, oLo[ie] + CTX.marcaVano, null);
      // El guion va SIN etiqueta: el vertice real del extremo esta en su centro y ya la
      // lleva, asi que repetirla solo agrandaria la traza sin agregar hover.
      capa.txt.push('', '', '');
      capa.cd.push(fid, fid, fid);
    }
  }

  // Las celdas de la ventana, indexadas por fid: {fid: [uiti, eventos, clase]}.
  function celdasDe(circ, ventana) {
    var col = ((CTX.celdas[circ] || [])[ventana]) || {fids: [], u: [], n: [], k: []};
    var mapa = {}, i;
    for (i = 0; i < col.fids.length; i++) {
      mapa[col.fids[i]] = [col.u[i], col.n[i], col.k[i]];
    }
    return mapa;
  }

  function dibujarMapaHistorico(gd) {
    var ventana = ventanaActual(), v = CTX.ventanas[ventana];
    var geo = geoDe(CIRC), den = geoDenso(CIRC), celdas = celdasDe(CIRC, ventana);
    var clases = [], marcadosClase = [], i, c;
    for (i = 0; i < 4; i++) { clases.push(capaVacia()); marcadosClase.push(capaVacia()); }
    var sinDato = capaVacia(), halo = capaVacia(), marcadoSinDato = capaVacia();

    for (i = 0; i < geo.fids.length; i++) {
      var fid = geo.fids[i], celda = celdas[fid];
      // Sin celda en la ventana no hay clase, y eso NO es el grupo mas bajo: es la
      // ausencia del dato. Mismo criterio que el tooltip del cuaderno.
      var clase = celda ? celda[2] : null;
      var etiqueta = '<b>Vano ' + fid + '</b><br>' + v.etiqueta + ': ' + v.periodo +
        '<br>Criticidad original: ' +
        (clase === null ? 'sin dato' : CTX.nombresGrupos[clase]) +
        '<br>UITI acumulado: ' + (celda ? celda[0].toLocaleString() : '0') +
        '<br>Eventos: ' + (celda ? celda[1] : 0) +
        (MARCADOS[fid] ? '<br>(marcado)' : '');
      var dLa = den.lat[i], dLo = den.lon[i], oLa = geo.lat[i], oLo = geo.lon[i];
      agregarVano(clase === null ? sinDato : clases[clase], fid, dLa, dLo, oLa, oLo, etiqueta);
      if (MARCADOS[fid]) {
        agregarVano(halo, fid, dLa, dLo, oLa, oLo, etiqueta);
        agregarVano(clase === null ? marcadoSinDato : marcadosClase[clase],
                    fid, dLa, dLo, oLa, oLo, etiqueta);
      }
    }

    var capas = clases.concat([sinDato, halo], marcadosClase, [marcadoSinDato]);
    var indices = CTX.idx.clases.concat([CTX.idx.sin_dato, CTX.idx.marcados],
                                        CTX.idx.marcados_clases, [CTX.idx.marcados_sin_dato]);
    Plotly.restyle(gd, {
      lat: capas.map(function (x) { return x.lat; }),
      lon: capas.map(function (x) { return x.lon; }),
      hovertext: capas.map(function (x) { return x.txt; }),
      customdata: capas.map(function (x) { return x.cd; }),
    }, indices);

    var tr = CTX.trafos[CIRC] || {lat: [], lon: []};
    var sw = CTX.switches[CIRC] || {lat: [], lon: []};
    Plotly.restyle(gd, {
      lat: [tr.lat, sw.lat], lon: [tr.lon, sw.lon],
      hovertext: [tr.lat.map(function () { return '<b>Transformador</b>'; }),
                  sw.lat.map(function () { return '<b>Interruptor / switch</b>'; })],
    }, [CTX.idx.trafos, CTX.idx.switches]);

    // El encuadre depende del CIRCUITO y no de la ventana: mover la ventana no tiene por
    // que reencuadrar el mapa.
    if (RECENTRADO !== CIRC && geo.bounds) {
      RECENTRADO = CIRC;
      encuadrar(gd);
    }
  }

  // Nube KMeans: el fondo ya viene dibujado y no depende de la seleccion. Aqui solo se
  // repinta el resaltado. Sin vanos marcados se resalta el circuito+ventana completo --
  // mismo grano al que caen el mapa simulado y el ranking, no un panel vacio.
  function dibujarNube(gd) {
    var ventana = ventanaActual();
    var col = ((CTX.celdas[CIRC] || [])[ventana]) || {fids: [], u: [], n: [], k: []};
    var lista = marcadosLista(), filtra = lista.length > 0;
    var x = [], y = [], color = [], txt = [], i;
    for (i = 0; i < col.fids.length; i++) {
      if (filtra && !MARCADOS[col.fids[i]]) { continue; }
      x.push(col.n[i]);
      y.push(col.u[i]);
      color.push(CTX.coloresGrupos[col.k[i]]);
      txt.push('<b>Vano ' + col.fids[i] + '</b><br>Eventos: ' + col.n[i].toLocaleString() +
               '<br>UITI acumulado: ' + col.u[i] +
               '<br>Grupo: ' + CTX.nombresGrupos[col.k[i]]);
    }
    Plotly.restyle(gd, {x: [x], y: [y], 'marker.color': [color], hovertext: [txt]},
                   [CTX.idx.nube_seleccion]);
  }

  // Evolucion: los primeros nCupos vanos marcados, en el orden de la geometria. Una
  // ventana sin celda va como null y NO como 0 -- un cero se leeria como "no hubo UITI",
  // y lo que paso es que no hubo medicion; `connectgaps:false` corta la linea ahi.
  function dibujarEvolucion(gd) {
    var geo = geoDe(CIRC), i, k;
    var elegidos = [];
    for (i = 0; i < geo.fids.length && elegidos.length < CTX.nCupos; i++) {
      if (MARCADOS[geo.fids[i]] && elegidos.indexOf(geo.fids[i]) < 0) {
        elegidos.push(geo.fids[i]);
      }
    }
    // El indice de cada ventana se arma UNA vez y no una por cupo: con 6 cupos y 11
    // ventanas eran 66 barridos del circuito completo en cada repintado.
    var porVentana = [];
    for (i = 0; i < CTX.ventanas.length; i++) { porVentana.push(celdasDe(CIRC, i)); }
    var xs = [], ys = [], txts = [];
    for (k = 0; k < CTX.nCupos; k++) {
      var fid = elegidos[k];
      if (!fid) { xs.push([]); ys.push([]); txts.push([]); continue; }
      var x = [], y = [], t = [];
      for (i = 0; i < CTX.ventanas.length; i++) {
        var celda = porVentana[i][fid];
        x.push(i);
        y.push(celda ? celda[0] : null);
        t.push(celda ? ('<b>Vano ' + fid + '</b><br>' + CTX.ventanas[i].etiqueta + ': ' +
                        CTX.ventanas[i].periodo + '<br>UITI: ' + celda[0] +
                        '<br>Eventos: ' + celda[1]) : '');
      }
      xs.push(x); ys.push(y); txts.push(t);
    }
    Plotly.restyle(gd, {x: xs, y: ys, hovertext: txts}, CTX.idx.evolucion);
  }

  // Violines: describen SOLO los vanos marcados. Sin marcados quedan VACIOS a proposito
  // (regla de 01.4): una distribucion sobre miles de vanos y una sobre tres se dibujan
  // igual, y nada en un violin las distingue -- caer al circuito completo cambiaria el
  // sujeto del panel en silencio.
  function dibujarViolines(gd) {
    var ventana = ventanaActual();
    var col = ((CTX.celdas[CIRC] || [])[ventana]) || {fids: [], u: [], n: [], k: []};
    var lista = marcadosLista();
    var uiti = [[], [], [], []], eventos = [[], [], [], []], i, maxEventos = 0;
    if (lista.length) {
      for (i = 0; i < col.fids.length; i++) {
        if (!MARCADOS[col.fids[i]]) { continue; }
        uiti[col.k[i]].push(col.u[i]);
        eventos[col.k[i]].push(col.n[i]);
        if (col.n[i] > maxEventos) { maxEventos = col.n[i]; }
      }
    }
    Plotly.restyle(gd, {y: uiti}, CTX.idx.violin_uiti);
    Plotly.restyle(gd, {y: eventos}, CTX.idx.violin_eventos);
    // Sin vanos marcados el eje lineal de eventos autoescala a [-1, 4] y muestra un
    // "-1 eventos" que no existe: el rango se fija a mano, igual que en el cuaderno.
    var relayout = {};
    relayout[CTX.ejes.violinEventosY + '.range'] = [0, maxEventos ? maxEventos * 1.15 : 5];
    Plotly.relayout(gd, relayout);
  }

  // La fila 2, el grafo y el ranking son SALIDA DEL MODELO: no se pueden recalcular sin
  // el kernel. Viajan como la foto de la ultima corrida del cuaderno y se borran en
  // cuanto la seleccion se mueve -- mostrar la corrida de otra seleccion es la confusion
  // que separa las dos filas (D2).
  function vigenciaSimulado(gd) {
    var ventana = ventanaActual();
    var vigente = !!SIM_VIGENTE && SIM_VIGENTE.circuito === CIRC &&
                  SIM_VIGENTE.ventana === ventana;
    var anotaciones = {}, indices, vacio;
    if (!vigente) {
      indices = CTX.idx.pred_clases.concat([
        CTX.idx.pred_sin_dato, CTX.idx.pred_marcados, CTX.idx.pred_trafos,
        CTX.idx.pred_switches]);
      vacio = indices.map(function () { return []; });
      Plotly.restyle(gd, {lat: vacio, lon: vacio, hovertext: vacio}, indices);
      Plotly.restyle(gd, {x: [[]], y: [[]], hovertext: [[]]}, [CTX.idx.ranking]);
      Plotly.restyle(gd, {x: [[], []], y: [[], []]},
                     [CTX.idx.grafo_aristas, CTX.idx.grafo_pesos]);
      Plotly.restyle(gd, {x: [[], []], y: [[], []], text: [[], []]}, CTX.idx.grafo_nodos);
      anotaciones['annotations[' + CTX.idxAnotSimulado + '].text'] =
        CTX.simulables[CIRC]
          ? 'Presiona <b>Simular</b> para este circuito y esta ventana.'
          : ('El simulador no viaja para <b>' + CIRC + '</b>.<br><sup>Este archivo trae ' +
             'las instancias de: ' + Object.keys(CTX.simulables).join(', ') + '.</sup>');
      anotaciones['annotations[' + CTX.idxAnotGrafo + '].text'] =
        'El grafo se estima al simular.';
    } else {
      anotaciones['annotations[' + CTX.idxAnotSimulado + '].text'] = '';
      anotaciones['annotations[' + CTX.idxAnotGrafo + '].text'] = '';
    }
    Plotly.relayout(gd, anotaciones);
  }

  // --- Simulador ----------------------------------------------------------------------
  // El MISMO modelo del cuaderno, corriendo aqui. `MIL.predecir` es la transcripcion de
  // `mil_web_export.predecir_numpy`, que esta clavada contra torch en pytest; medido
  // contra el modelo real, u-hat coincide con 9e-6 de error relativo y ninguna clase
  // cambia. Los pesos se hidratan una sola vez, la primera vez que se simula.
  var PESOS = null;
  var X_CIRCUITO = {};          // cache de la matriz de instancias por circuito
  var VALORES_KNOB = {};        // knob id -> valor elegido
  var TAREA_RANKING = null;     // el barrido de relevancia, troceado

  function pesosMil() {
    if (!PESOS) { PESOS = MIL.hidratar(CTX.pesos); }
    return PESOS;
  }

  function simulable() { return !!CTX.simulables[CIRC]; }

  function instanciasDe(circ) {
    if (!X_CIRCUITO[circ]) { X_CIRCUITO[circ] = MIL.desB64(CTX.simulables[circ].X); }
    return X_CIRCUITO[circ];
  }

  // Las bolsas de la seleccion activa, con sus instancias copiadas a una matriz propia.
  // Sin vanos marcados el grano es el circuito completo en esa ventana -- el mismo al
  // que caen el mapa, el ranking y la nube, para que los cuatro paneles no describan
  // conjuntos distintos.
  function seleccionarBolsas() {
    if (!simulable()) { return null; }
    var sim = CTX.simulables[CIRC], ventana = ventanaActual();
    var filtra = marcadosLista().length > 0, elegidas = [], b;
    for (b = 0; b < sim.fid.length; b++) {
      if (sim.ventana[b] !== ventana) { continue; }
      if (filtra && !MARCADOS[sim.fid[b]]) { continue; }
      elegidas.push(b);
    }
    if (!elegidas.length) { return null; }
    var nCols = CTX.nFeatures, nInst = 0, i;
    for (i = 0; i < elegidas.length; i++) { nInst += sim.conteo[elegidas[i]]; }
    var fuente = instanciasDe(CIRC);
    var X = MIL.mat(nInst, nCols);
    var ib = new Int32Array(nInst), nObs = new Float64Array(elegidas.length);
    var fila = 0;
    for (i = 0; i < elegidas.length; i++) {
      b = elegidas[i];
      nObs[i] = sim.conteo[b];
      var desde = sim.inicio[b] * nCols, hasta = desde + sim.conteo[b] * nCols;
      X.d.set(fuente.d.subarray(desde, hasta), fila * nCols);
      for (var r = 0; r < sim.conteo[b]; r++) { ib[fila + r] = i; }
      fila += sim.conteo[b];
    }
    return {X: X, ib: ib, nBags: elegidas.length, nObs: nObs,
            fid: elegidas.map(function (k) { return sim.fid[k]; })};
  }

  // Escribe el valor de cada control en TODAS las instancias de sus columnas, que es lo
  // que hace `aplicar_overrides_instancias`. Una familia climatica mueve sus 12 rezagos
  // a la vez: es un control, no doce.
  function aplicarOverrides(X, overrides) {
    for (var k = 0; k < overrides.length; k++) {
      var indices = overrides[k].indices, valor = overrides[k].valor;
      for (var i = 0; i < X.m; i++) {
        var base = i * X.n;
        for (var j = 0; j < indices.length; j++) { X.d[base + indices[j]] = valor; }
      }
    }
  }

  function overridesActivos() {
    var salida = [], i;
    for (i = 0; i < CTX.knobs.length; i++) {
      var knob = CTX.knobs[i];
      if (Object.prototype.hasOwnProperty.call(VALORES_KNOB, knob.id)) {
        salida.push({indices: knob.indices, valor: VALORES_KNOB[knob.id], label: knob.label});
      }
    }
    return salida;
  }

  // Riesgo ordinal medio de la seleccion: `mean(softmax(-d^2) . [0..3])`, la misma
  // cantidad que mide `_riesgo_ordinal`. Es la distribucion SUAVE a proposito -- hace
  // que la diferencia entre escenarios sea un numero continuo y no un escalon.
  function riesgoOrdinal(nObs, u, geometria) {
    var offset = geometria.offset.d, escala = geometria.scale.d;
    var centroides = geometria.centroides, nC = centroides.m, total = 0, i, c;
    for (i = 0; i < u.length; i++) {
      var x0 = geometria.logs[0] ? Math.log10(Math.max(nObs[i], 1e-6)) : nObs[i];
      var x1 = geometria.logs[1] ? Math.log10(Math.max(u[i], 1e-6)) : u[i];
      var z0 = (x0 - offset[0]) / escala[0], z1 = (x1 - offset[1]) / escala[1];
      var d2 = [], maximo = -Infinity;
      for (c = 0; c < nC; c++) {
        var a = z0 - centroides.d[c * 2], b = z1 - centroides.d[c * 2 + 1];
        d2.push(-(a * a + b * b));
        if (d2[c] > maximo) { maximo = d2[c]; }
      }
      var suma = 0, esperado = 0;
      for (c = 0; c < nC; c++) { d2[c] = Math.exp(d2[c] - maximo); suma += d2[c]; }
      for (c = 0; c < nC; c++) { esperado += (d2[c] / suma) * c; }
      total += esperado;
    }
    return u.length ? total / u.length : 0;
  }

  function pintarMapaSimulado(gd, sel, claseSim) {
    var geo = geoDe(CIRC), den = geoDenso(CIRC), v = CTX.ventanas[ventanaActual()];
    var porFid = {}, i;
    for (i = 0; i < sel.fid.length; i++) { porFid[sel.fid[i]] = claseSim[i]; }
    var capas = [], sinDato = capaVacia();
    for (i = 0; i < 4; i++) { capas.push(capaVacia()); }
    for (i = 0; i < geo.fids.length; i++) {
      var fid = geo.fids[i];
      var clase = Object.prototype.hasOwnProperty.call(porFid, fid) ? porFid[fid] : null;
      var etiqueta = '<b>Vano ' + fid + '</b><br>' + v.etiqueta + ': ' + v.periodo +
        '<br>Criticidad simulada: ' +
        (clase === null ? 'sin evento / no simulado' : CTX.nombresGrupos[clase]);
      agregarVano(clase === null ? sinDato : capas[clase], fid,
                  den.lat[i], den.lon[i], geo.lat[i], geo.lon[i], etiqueta);
    }
    var todas = capas.concat([sinDato]);
    var indices = CTX.idx.pred_clases.concat([CTX.idx.pred_sin_dato]);
    // `customdata` viaja junto a lat/lon, igual que en el mapa de arriba y que en
    // `_volcar_capa` del cuaderno: las cuatro columnas tienen que medir lo mismo o
    // Plotly desalinea el resto de la traza, y sin ella un tramo del mapa simulado no
    // sabe a que vano pertenece.
    Plotly.restyle(gd, {
      lat: todas.map(function (x) { return x.lat; }),
      lon: todas.map(function (x) { return x.lon; }),
      hovertext: todas.map(function (x) { return x.txt; }),
      customdata: todas.map(function (x) { return x.cd; }),
      showlegend: indices.map(function () { return true; }),
    }, indices);
    var tr = CTX.trafos[CIRC] || {lat: [], lon: []};
    var sw = CTX.switches[CIRC] || {lat: [], lon: []};
    Plotly.restyle(gd, {
      lat: [tr.lat, sw.lat], lon: [tr.lon, sw.lon],
      hovertext: [tr.lat.map(function () { return '<b>Transformador</b>'; }),
                  sw.lat.map(function () { return '<b>Interruptor / switch</b>'; })],
    }, [CTX.idx.pred_trafos, CTX.idx.pred_switches]);
  }

  // El grafo reconstruido de la seleccion: `mean_vano(compuerta) * peso fijo`. Se ANULA
  // cuando las compuertas no varian entre vanos -- dibujarlo entonces seria presentar el
  // grafo experto fijo como si lo hubiera estimado esta seleccion.
  function pintarGrafoSimulado(gd, compuertas) {
    var pesos = pesosMil(), nA = compuertas.n, nB = compuertas.m, e, i;
    var mensaje = '', trazas = null;
    var rango = MIL.rangoEfectivo(compuertas);
    var casiConstante = true;
    for (e = 0; e < nA; e++) {
      var media = 0;
      for (i = 0; i < nB; i++) { media += compuertas.d[i * nA + e]; }
      media /= nB;
      var varianza = 0;
      for (i = 0; i < nB; i++) {
        var dif = compuertas.d[i * nA + e] - media;
        varianza += dif * dif;
      }
      if (nB > 1 && Math.sqrt(varianza / (nB - 1)) >= 1e-6) { casiConstante = false; }
    }
    if (casiConstante || rango <= 1 + 1e-9) {
      mensaje = 'Grafo no estimable: las compuertas no varian entre los ' + nB +
        ' vanos de la seleccion.<br><sup>Hacen falta al menos 3 vanos con ' +
        'comportamiento distinto.</sup>';
    } else {
      // Disposicion circular sobre las variables que participan de alguna arista,
      // igual que `trazas_grafo`.
      var filas = pesos.aristaFilas.d, cols = pesos.aristaColumnas.d;
      var valores = pesos.aristaValores.d;
      var participantes = [], visto = {};
      for (e = 0; e < nA; e++) {
        [filas[e], cols[e]].forEach(function (n) {
          if (!visto[n]) { visto[n] = true; participantes.push(n); }
        });
      }
      participantes.sort(function (a, b) { return a - b; });
      var pos = {};
      participantes.forEach(function (n, k) {
        var ang = 2 * Math.PI * k / participantes.length;
        pos[n] = [Math.cos(ang), Math.sin(ang)];
      });
      var ax = [], ay = [], px = [], py = [], pw = [], ph = [];
      for (e = 0; e < nA; e++) {
        var suma = 0;
        for (i = 0; i < nB; i++) { suma += compuertas.d[i * nA + e]; }
        var peso = (suma / nB) * valores[e];
        if (peso === 0) { continue; }
        var o = pos[filas[e]], d = pos[cols[e]];
        ax.push(o[0], d[0], null);
        ay.push(o[1], d[1], null);
        px.push((o[0] + d[0]) / 2);
        py.push((o[1] + d[1]) / 2);
        pw.push(peso);
        ph.push('<b>' + CTX.featuresMil[filas[e]] + ' &#8594; ' +
                CTX.featuresMil[cols[e]] + '</b><br>Peso reconstruido: ' +
                peso.toPrecision(4));
      }
      trazas = {ax: ax, ay: ay, px: px, py: py, pw: pw, ph: ph, participantes: participantes,
                pos: pos};
    }

    if (!trazas) {
      Plotly.restyle(gd, {x: [[], []], y: [[], []]},
                     [CTX.idx.grafo_aristas, CTX.idx.grafo_pesos]);
      Plotly.restyle(gd, {x: [[], []], y: [[], []], text: [[], []]}, CTX.idx.grafo_nodos);
    } else {
      Plotly.restyle(gd, {x: [trazas.ax], y: [trazas.ay]}, [CTX.idx.grafo_aristas]);
      var maximo = Math.max.apply(null, trazas.pw.map(Math.abs)) || 1;
      Plotly.restyle(gd, {
        x: [trazas.px], y: [trazas.py], hovertext: [trazas.ph],
        'marker.size': [trazas.pw.map(function (p) { return 4 + 10 * Math.abs(p) / maximo; })],
        'marker.color': [trazas.pw],
      }, [CTX.idx.grafo_pesos]);
      // Un trazo de nodos por MODALIDAD: el color dice a que modo pertenece la variable.
      for (var m = 0; m < CTX.idx.grafo_nodos.length; m++) {
        var columnas = CTX.columnasModalidad[m] || [];
        var cuales = trazas.participantes.filter(function (n) {
          return columnas.indexOf(n) >= 0;
        });
        Plotly.restyle(gd, {
          x: [cuales.map(function (n) { return trazas.pos[n][0]; })],
          y: [cuales.map(function (n) { return trazas.pos[n][1]; })],
          text: [cuales.map(function (n) { return CTX.featuresMil[n]; })],
          textposition: [cuales.map(function (n) {
            return trazas.pos[n][0] >= 0 ? 'middle right' : 'middle left'; })],
          hovertext: [cuales.map(function (n) {
            return '<b>' + CTX.featuresMil[n] + '</b><br>Modo: ' + CTX.modalidades[m]; })],
        }, [CTX.idx.grafo_nodos[m]]);
      }
    }
    var anot = {};
    anot['annotations[' + CTX.idxAnotGrafo + '].text'] = mensaje;
    Plotly.relayout(gd, anot);
  }

  // Barrido min/max: `1 + 2 x knobs numericos` pasadas. Va TROCEADO con setTimeout y no
  // en un bucle cerrado: con la seleccion mas grande medida son 45 forwards, y hacerlos
  // de corrido congela la pestania sin decir nada. Asi la barra se llena a la vista.
  function barridoRelevancia(gd, sel, riesgoBase) {
    if (TAREA_RANKING) { clearTimeout(TAREA_RANKING); TAREA_RANKING = null; }
    var numericos = CTX.knobs.filter(function (k) { return k.kind === 'numeric' && k.bounds; });
    var filas = [], indice = 0;
    var pesos = pesosMil();

    function paso() {
      if (indice >= numericos.length) {
        TAREA_RANKING = null;
        estado('Listo. ' + filas.length + ' variables en el barrido de relevancia.');
        return;
      }
      var knob = numericos[indice++];
      var deltas = [];
      [knob.bounds[0], knob.bounds[1]].forEach(function (valor) {
        var X = {d: sel.X.d.slice(), m: sel.X.m, n: sel.X.n};
        aplicarOverrides(X, [{indices: knob.indices, valor: valor}]);
        var r = MIL.predecir(pesos, X, sel.ib, sel.nBags);
        deltas.push(riesgoOrdinal(sel.nObs, r.u, pesos.geometria) - riesgoBase);
      });
      filas.push({label: knob.label,
                  magnitud: Math.max(Math.abs(deltas[0]), Math.abs(deltas[1]))});
      pintarRanking(gd, filas);
      estado('Relevancia: ' + indice + ' de ' + numericos.length + ' variables...');
      TAREA_RANKING = setTimeout(paso, 0);
    }
    TAREA_RANKING = setTimeout(paso, 0);
  }

  // Softmax sobre las magnitudes crudas (temperatura 1), igual que `normalizar_softmax`:
  // una participacion se lee sin conocer las unidades del modelo, una sensibilidad
  // min-max de 0,0143 no.
  function pintarRanking(gd, filas) {
    var ordenadas = filas.slice().sort(function (a, b) { return a.magnitud - b.magnitud; });
    var maximo = -Infinity, i;
    for (i = 0; i < ordenadas.length; i++) {
      if (ordenadas[i].magnitud > maximo) { maximo = ordenadas[i].magnitud; }
    }
    var exp = ordenadas.map(function (f) { return Math.exp(f.magnitud - maximo); });
    var suma = exp.reduce(function (a, b) { return a + b; }, 0) || 1;
    var relevancia = exp.map(function (e) { return e / suma; });
    var tope = Math.max.apply(null, relevancia.concat([0]));
    Plotly.restyle(gd, {
      y: [ordenadas.map(function (f) { return f.label; })],
      x: [relevancia],
      hovertext: [ordenadas.map(function (f, k) {
        return '<b>' + f.label + '</b><br>Relevancia: ' + (relevancia[k] * 100).toFixed(1) +
               '%<br>Sensibilidad min-max: ' + f.magnitud.toPrecision(4);
      })],
    }, [CTX.idx.ranking]);
    var rl = {};
    rl[CTX.ejes.rankingX + '.range'] = [0, tope > 0 ? tope * 1.15 : 0.25];
    Plotly.relayout(gd, rl);
  }

  function estado(texto) { d.getElementById('sim-estado').innerHTML = texto; }

  function simular() {
    var gd = gdListo();
    if (!gd) { return; }
    if (!simulable()) {
      estado('<b>' + CIRC + '</b> no viaja en este archivo. Se exportaron las instancias ' +
             'de: ' + Object.keys(CTX.simulables).join(', ') +
             '. Ejecuta el cuaderno con ese circuito activo, o agrega ese circuito a ' +
             '<code>CIRCUITOS_SIMULABLES</code>.');
      return;
    }
    var sel = seleccionarBolsas();
    if (!sel) {
      estado('Sin bolsas (vano x ventana) para esta seleccion.');
      return;
    }
    estado('Simulando ' + sel.nBags + ' bolsas (' + sel.X.m + ' eventos)...');
    // Un tick antes del computo pesado, para que el navegador pinte el "Simulando...".
    setTimeout(function () {
      var t0 = (new Date()).getTime();
      var pesos = pesosMil();
      var base = MIL.predecir(pesos, sel.X, sel.ib, sel.nBags);
      var Xsim = {d: sel.X.d.slice(), m: sel.X.m, n: sel.X.n};
      var overrides = overridesActivos();
      aplicarOverrides(Xsim, overrides);
      var sim = overrides.length ? MIL.predecir(pesos, Xsim, sel.ib, sel.nBags) : base;
      var claseBase = MIL.clase(sel.nObs, base.u, pesos.geometria);
      var claseSim = MIL.clase(sel.nObs, sim.u, pesos.geometria);
      var cambian = 0;
      for (var k = 0; k < claseBase.length; k++) {
        if (claseBase[k] !== claseSim[k]) { cambian++; }
      }
      SIM_VIGENTE = {circuito: CIRC, ventana: ventanaActual()};
      pintarMapaSimulado(gd, sel, claseSim);
      // El SEGUNDO mapa suele quedar bajo el pliegue, y MapLibre no termina de cargar su
      // estilo hasta que hace falta: un restyle disparado antes se pierde en silencio
      // ("Style is not done loading", 26 veces en la sonda headless). Se repite una vez,
      // es idempotente, y es el mismo apanio que la fila 1 ya usaba al arrancar.
      setTimeout(function () {
        var g2 = gdListo();
        if (g2 && SIM_VIGENTE && SIM_VIGENTE.circuito === CIRC &&
            SIM_VIGENTE.ventana === ventanaActual()) {
          pintarMapaSimulado(g2, sel, claseSim);
        }
      }, 900);
      // El grafo se estima sobre las compuertas OBSERVADAS: describe a estos vanos, no
      // al escenario hipotetico.
      pintarGrafoSimulado(gd, base.compuertas);
      Plotly.relayout(gd, (function () {
        var a = {}; a['annotations[' + CTX.idxAnotSimulado + '].text'] = ''; return a;
      })());
      var ms = (new Date()).getTime() - t0;
      estado((ms / 1000).toFixed(2) + ' s | MIL sobre ' + sel.nBags + ' bolsas (' +
             sel.X.m + ' eventos) | ' + overrides.length + ' variables aplicadas | ' +
             cambian + ' vanos cambian de clase | calculando relevancia...');
      barridoRelevancia(gd, sel, riesgoOrdinal(sel.nObs, base.u, pesos.geometria));
    }, 30);
  }

  // --- Controles del simulador --------------------------------------------------------
  function poblarKnobs() {
    var html = '', i;
    for (i = 0; i < CTX.knobs.length; i++) {
      var k = CTX.knobs[i];
      html += '<label><input type="checkbox" data-knob="' + k.id + '">' + k.label + '</label>';
    }
    d.getElementById('sim-knobs').innerHTML = html;
  }

  function reconstruirControles() {
    var caja = d.getElementById('sim-controles'), html = '', i;
    var marcados = Array.prototype.slice.call(
      d.querySelectorAll('#sim-knobs input:checked')).map(function (c) {
        return c.getAttribute('data-knob'); });
    var nuevos = {};
    for (i = 0; i < CTX.knobs.length; i++) {
      var k = CTX.knobs[i];
      if (marcados.indexOf(k.id) < 0) { continue; }
      // El valor sobrevive si el control ya estaba: destildar y volver a tildar no tiene
      // por que perder lo que el usuario habia puesto.
      nuevos[k.id] = Object.prototype.hasOwnProperty.call(VALORES_KNOB, k.id)
        ? VALORES_KNOB[k.id]
        : (k.kind === 'numeric' ? (k.bounds[0] + k.bounds[1]) / 2 : k.opciones[0].valor);
      html += '<div class="sim-knob"><span>' + k.label + '</span>';
      if (k.kind === 'numeric') {
        html += '<input type="range" data-knob="' + k.id + '" min="' + k.bounds[0] +
                '" max="' + k.bounds[1] + '" step="' + (k.step || (k.bounds[1] - k.bounds[0]) / 100) +
                '" value="' + nuevos[k.id] + '">' +
                '<span class="sim-valor" id="sim-val-' + i + '">' +
                nuevos[k.id].toPrecision(4) + '</span>';
      } else {
        html += '<select data-knob="' + k.id + '">';
        for (var j = 0; j < k.opciones.length; j++) {
          html += '<option value="' + k.opciones[j].valor + '"' +
                  (k.opciones[j].valor === nuevos[k.id] ? ' selected' : '') + '>' +
                  k.opciones[j].etiqueta + '</option>';
        }
        html += '</select>';
      }
      html += '</div>';
    }
    VALORES_KNOB = nuevos;
    caja.innerHTML = html;
  }

  d.getElementById('sim-knobs').addEventListener('change', reconstruirControles);
  d.getElementById('sim-controles').addEventListener('input', function (e) {
    var id = e.target && e.target.getAttribute('data-knob');
    if (!id) { return; }
    VALORES_KNOB[id] = parseFloat(e.target.value);
    var valor = e.target.parentNode.querySelector('.sim-valor');
    if (valor) { valor.textContent = VALORES_KNOB[id].toPrecision(4); }
  });
  d.getElementById('sim-correr').addEventListener('click', simular);

  // --- Lista de vanos ---------------------------------------------------------------
  function poblarVanos() {
    var caja = d.getElementById('sim-vanos');
    var lista = CTX.vanos[CIRC] || [];
    var html = '', i;
    for (i = 0; i < lista.length; i++) {
      var fid = String(lista[i]);
      html += '<label><input type="checkbox" data-fid="' + fid + '"' +
              (MARCADOS[fid] ? ' checked' : '') + '>' + fid + '</label>';
    }
    caja.innerHTML = html;
    actualizarConteo();
  }

  function actualizarConteo() {
    var n = marcadosLista().length;
    d.getElementById('sim-conteo').textContent =
      n === 0 ? 'ninguno (los violines describen solo lo marcado)'
              : n + ' marcado' + (n === 1 ? '' : 's') +
                (n > CTX.nCupos ? ' -- la evolucion dibuja los primeros ' + CTX.nCupos : '');
  }

  function sincronizarCasilla(fid) {
    var casilla = d.querySelector('#sim-vanos input[data-fid="' + fid + '"]');
    if (casilla) { casilla.checked = !!MARCADOS[fid]; }
  }

  // --- Repintados -------------------------------------------------------------------
  function gdListo() {
    var gd = d.getElementById(CTX.div);
    return (gd && gd._fullLayout) ? gd : null;
  }

  // Lo que depende de la SELECCION de vanos: mapa (halo y color de marcado), nube,
  // evolucion y violines. La fila 2 no se toca -- marcar un vano no invalida la corrida.
  function repintarSeleccion() {
    var gd = gdListo();
    if (!gd) { return setTimeout(repintarSeleccion, 120); }
    dibujarMapaHistorico(gd);
    dibujarNube(gd);
    dibujarEvolucion(gd);
    dibujarViolines(gd);
    actualizarConteo();
  }

  // Lo que depende del circuito o de la ventana: ademas de todo lo anterior, revisa si la
  // foto de la simulacion sigue correspondiendo a lo que se esta mirando.
  function repintarTodo() {
    var gd = gdListo();
    if (!gd) { return setTimeout(repintarTodo, 120); }
    var v = CTX.ventanas[ventanaActual()];
    d.getElementById('sim-ventana-txt').textContent = v.etiqueta + ': ' + v.periodo;
    Plotly.relayout(gd, {'title.text': 'Simulador Criticidad -- ' + CIRC});
    dibujarMapaHistorico(gd);
    dibujarNube(gd);
    dibujarEvolucion(gd);
    dibujarViolines(gd);
    vigenciaSimulado(gd);
    d.getElementById('sim-aviso').textContent =
      CIRC + ' -- ' + (CTX.vanos[CIRC] || []).length + ' vanos con eventos en alguna de las ' +
      CTX.ventanas.length + ' ventanas.';
  }

  // --- Controles ---------------------------------------------------------------------
  d.getElementById('sim-circuito').addEventListener('change', function (e) {
    CIRC = e.target.value;
    MARCADOS = {};              // la seleccion es por circuito: los fids no se comparten
    poblarVanos();
    repintarTodo();
    estado(simulable() ? 'Circuito cambiado: vuelve a simular.'
                       : 'Este archivo no trae las instancias de ' + CIRC + '.');
  });

  d.getElementById('sim-ventana').addEventListener('input', function () {
    repintarTodo();
    estado(simulable() ? 'Ventana cambiada: vuelve a simular.'
                       : 'Este archivo no trae las instancias de ' + CIRC + '.');
  });

  d.getElementById('sim-vanos').addEventListener('change', function (e) {
    var fid = e.target && e.target.getAttribute('data-fid');
    if (!fid) { return; }
    if (e.target.checked) { MARCADOS[fid] = true; } else { delete MARCADOS[fid]; }
    repintarSeleccion();
  });

  d.getElementById('sim-marcar-todos').addEventListener('click', function () {
    (CTX.vanos[CIRC] || []).forEach(function (f) { MARCADOS[String(f)] = true; });
    poblarVanos();
    repintarSeleccion();
  });

  d.getElementById('sim-desmarcar').addEventListener('click', function () {
    MARCADOS = {};
    poblarVanos();
    repintarSeleccion();
  });

  // Clic en el mapa: alterna el vano, igual que en el cuaderno. El fid sale de
  // `customdata` y NO del indice del punto -- los tramos viajan concatenados con un
  // separador, asi que ese indice se mueve con la ventana. Solo el mapa de la fila 1: la
  // fila 2 es la salida del modelo, no un control (D2).
  function conectarClic() {
    var gd = gdListo();
    if (!gd || typeof gd.on !== 'function') { return setTimeout(conectarClic, 120); }
    var seleccionables = CTX.idx.clases.concat([
      CTX.idx.sin_dato, CTX.idx.marcados, CTX.idx.marcados_sin_dato],
      CTX.idx.marcados_clases);
    gd.on('plotly_click', function (e) {
      var p = e.points && e.points[0];
      if (!p || seleccionables.indexOf(p.curveNumber) < 0) { return; }
      var fid = p.customdata;
      if (fid === undefined || fid === null) { return; }
      fid = String(fid);
      if (MARCADOS[fid]) { delete MARCADOS[fid]; } else { MARCADOS[fid] = true; }
      sincronizarCasilla(fid);
      repintarSeleccion();
    });
  }

  poblarVanos();
  poblarKnobs();
  reconstruirControles();
  conectarClic();
  repintarTodo();
  estado(simulable()
    ? 'Elige variables (opcional) y presiona Simular.'
    : 'Este archivo no trae las instancias de ' + CIRC + '.');
  // MapLibre inicializa de forma asincrona: un restyle disparado antes de que el subplot
  // de mapa este listo se pierde en silencio. Se repite el dibujado un par de veces; es
  // idempotente.
  [700, 2000].forEach(function (ms) {
    setTimeout(function () {
      var gd = gdListo();
      if (!gd) { return; }
      dibujarMapaHistorico(gd);
      // El PRIMER encuadre puede haber corrido antes de que MapLibre montara su canvas,
      // con el tamanio aproximado del dominio. Aqui ya hay medida real, asi que se rehace
      // -- salvo que el usuario ya haya movido el mapa.
      if (camaraSinTocar(gd)) { encuadrar(gd); }
    }, ms);
  });

  // La figura es responsive: al cambiar el tamanio de la ventana el mapa cambia de
  // pixeles y el zoom que encuadraba deja de encuadrar.
  var temporizadorResize = null;
  window.addEventListener('resize', function () {
    clearTimeout(temporizadorResize);
    temporizadorResize = setTimeout(function () {
      var gd = gdListo();
      if (gd && camaraSinTocar(gd)) { encuadrar(gd); }
    }, 350);
  });
})();
</script>
'''
if EXPORTAR_PANEL_WEB:
    PANEL_JS_WEB = PANEL_JS_WEB_TEMPLATE.replace(
        '__CTX_JSON__', json.dumps(CONTEXTO_WEB, separators=(',', ':')))

    # `include_plotlyjs=True` embebe la libreria en la misma salida: el archivo funciona sin
    # conexion y sin el cuaderno. `default_width='100%'` solo surte efecto porque la figura ya
    # NO lleva `width`, y `responsive` la recalcula al cambiar el tamanio de la ventana.
        # La COPIA que viaja al navegador pierde el ancho fijo de la celda: ahi si se quiere
    # que la figura se estire a la pantalla y se reajuste al cambiar el tamanio de la
    # ventana. `default_width='100%'` solo surte efecto porque `width` quedo en None.
    _figura_web = go.Figure(fig)
    _figura_web.layout.width = None
    _figura_web.layout.autosize = True
    FIGURA_HTML_WEB = pio.to_html(_figura_web, include_plotlyjs=True, full_html=False,
                                  div_id=DIV_FIGURA, default_width='100%',
                                  config={'responsive': True})
    PANEL_COMPLETO_WEB = PANEL_HTML_WEB + FIGURA_HTML_WEB + PANEL_JS_WEB


    def exportar_y_abrir(html_panel, *, abrir=True):
        """Escribe el panel autocontenido y, SOLO en local, lo abre en el navegador.

        En Databricks nunca abre nada (ver `EN_DATABRICKS`): escribe el HTML en el Volume e
        imprime como bajarlo. Falla suave y no aborta el cuaderno si el Volume rechaza la
        escritura -- alla el panel es un extra que se puede descargar, no el entregable, y
        tumbar una corrida entera por un archivo opcional seria peor que avisar. En local si
        propaga el error: ahi el archivo ES el entregable.
        """
        destino = ROOT / 'reports' / 'paneles' / '06_uiti_vano_explicabilidad_simulador.html'
        try:
            destino.parent.mkdir(parents=True, exist_ok=True)
        except OSError as exc:
            if not EN_DATABRICKS:
                raise
            print(f'AVISO: no se pudo crear {destino.parent} ({exc}). El panel web no se '
                  'escribe; la app de widgets de la celda anterior sigue funcionando.')
            return None
        documento = (
            '<!doctype html>\n<html lang="es">\n<head>\n<meta charset="utf-8">\n'
            '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
            '<title>Simulador Criticidad por vano</title>\n'
            # margin 0 + un div al 100%: sin esto el navegador deja el margen por defecto del
            # body y la figura no llega a los bordes de la pantalla.
            '<style>html,body{margin:0;padding:12px;box-sizing:border-box;'
            'font-family:system-ui,-apple-system,"Segoe UI",sans-serif;}'
            f'#{DIV_FIGURA}{{width:100%;}}</style>\n</head>\n<body>\n'
            + html_panel + '\n</body>\n</html>\n'
        )
        try:
            destino.write_text(documento, encoding='utf-8')
        except OSError as exc:
            if not EN_DATABRICKS:
                raise
            print(f'AVISO: no se pudo escribir {destino} ({exc}). La app de widgets de la '
                  'celda anterior sigue funcionando.')
            return None
        mb = destino.stat().st_size / 1024 ** 2

        if EN_DATABRICKS:
            print(f'panel autocontenido escrito en {destino} ({mb:,.1f} MB)')
            print('Databricks: no se abre ningun navegador -- el driver no tiene uno. La app '
                  'de widgets de la celda anterior es la interfaz de este cuaderno.')
            # El `databricks fs cp` solo aplica si el destino cayo en un Volume, que es donde
            # lo pone la reescritura de arranque de /subir-notebooks-databricks. Si alguien
            # corre con otro ROOT, imprimir ese comando seria mandarlo a una ruta que la CLI
            # no resuelve.
            if str(destino).startswith('/Volumes/'):
                print(f'  para verlo a ancho completo, descargalo y abrelo en tu maquina:\n'
                      f'    databricks fs cp dbfs:{destino} ./06_panel.html')
            else:
                print(f'  ROOT no es un Volume: el archivo quedo en {destino}, descargalo por '
                      'donde corresponda a ese almacenamiento.')
            return destino

        print(f'panel autocontenido escrito en {destino.relative_to(ROOT)} ({mb:,.1f} MB)')
        if abrir:
            import webbrowser

            webbrowser.open(destino.resolve().as_uri())
            print(f'abriendo en el navegador por defecto -- pesa {mb:,.0f} MB, asi que la '
                  'primera carga puede tardar unos segundos')
        else:
            print('ABRIR_EN_NAVEGADOR = False: no se abre nada, el archivo queda escrito')
        return destino


    RUTA_PANEL_WEB = exportar_y_abrir(PANEL_COMPLETO_WEB, abrir=ABRIR_EN_NAVEGADOR)

else:
    RUTA_PANEL_WEB = None
    print('EXPORTAR_PANEL_WEB = False: no se escribe ningun HTML y no se abre el '
          'navegador.\n  El tablero completo -- circuito, ventana, vanos, variables y '
          '"Simular" -- esta en la app de widgets de la celda anterior.\n  Cambialo a True '
          'si ademas quieres el archivo autocontenido para compartir.')


## Como leerlo

**Los dos mapas.** Arriba, la criticidad historica; abajo, la simulada. Comparten encuadre
a proposito -- la comparacion solo se sostiene si miran la misma geografia -- pero nunca
leyenda ni titulo, porque mezclarlos invita a leer una prediccion como un hecho observado.

**Negro = sin evento**, en los dos mapas. Un vano sin eventos en la ventana no tiene clase,
y la ausencia no es el grupo mas bajo. En el mapa simulado el negro cubre ademas lo que
quedo fuera de la seleccion.

**Marcar un vano** se hace con su casilla o tocandolo en el mapa de arriba: las dos vias
son el mismo estado, porque el clic alterna la casilla. Solo la fila 1 acepta clic; la
fila 2 es la salida del modelo, no un control, y marcar desde ahi mezclaria "lo que elegi"
con "lo que el modelo predijo" sobre la misma superficie. Un vano marcado se dibuja con el
color de SU clase sobre un halo blanco -- no con un color plano de "seleccionado", que
congelaria lo que se ve cuando la ventana cambia la clase por debajo.

**"Simular" es el unico disparador** y produce las tres salidas en el mismo trabajo,
aplicando solo las variables elegidas en el panel. Cada variable aparece como un control
-- deslizador si es numerica, lista si es categorica -- y una familia climatica
(precipitacion, temperatura, rafaga y viento) mueve sus 12 rezagos horarios de una vez.
El mapa simulado **no existe hasta que se presiona**: antes solo muestra el aviso, porque
un mapa pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma forma que un
resultado.

**Cambiar circuito o ventana descarta la ultima simulacion.** La fila 2, el grafo y la
importancia se vacian: mostrar la corrida de otra seleccion es la misma confusion que
todo lo anterior evita.

**"Importancia Variables"** (fila 3, columna 1) no es SHAP: es un barrido min-max sobre el
mismo modelo y las mismas bolsas que pinta el mapa, con costo `1 + 2 x controles
numericos` pasadas. Se muestra normalizado con softmax, asi que las barras suman 100% y se
leen como participacion relativa; la magnitud cruda queda en el hover. Cuidado con una
propiedad del softmax: cuando todas las magnitudes son parecidas las barras se reparten
casi parejo, y eso significa "ninguna variable domina", no un error. Con este modelo
ocurre: el barrido mueve el riesgo ordinal entre 0,02 y 0,08, asi que las 22 barras quedan
cerca del 4,5% del reparto uniforme.

**"Grupos KMeans de vanos"** (fila 3, columna 2) es la nube de 04: cada punto es una celda
(vano, ventana) en el plano `(eventos, UITI acumulado)`, con las fronteras de Voronoi
debajo y lo marcado resaltado encima. El fondo es una muestra fija de 20.000 celdas de las
111 mil -- el resto es sobredibujo, y mandarlas todas supera el limite de datos por segundo
del kernel, que descarta el mensaje y deja la figura sin dibujar. La frontera se calcula
con la MISMA funcion que clasifica a los vanos, no con una regla propia. Aqui se ve directo
por que mover la ventana cambia la clase de un vano: su punto se desplaza por el plano.

**"Grafo reconstruido"** (fila 3, columna 3) es el grafo experto tal como lo usa la
seleccion: `media_vanos(compuerta) x peso_fijo` por arista, en disposicion circular. Cada
nodo lleva su variable y el color de su modo -- `climaticos` o `estructurales` --, paleta
deliberadamente ajena a la de los grupos KMeans. Se **anula** cuando las compuertas no
varian entre los vanos, lo que incluye por construccion cualquier seleccion de menos de 3
vanos: dibujarlo igual seria presentar el grafo experto fijo como si lo hubiera estimado
esta seleccion.

**La fila 4 describe a los vanos marcados.** A la izquierda, la evolucion de UITI de los
primeros 6 a lo largo de las ventanas, con un corte donde el vano no tiene celda -- ese
hueco va vacio y no en cero, porque no es "no hubo UITI" sino "no hubo medicion". A la
derecha, el reparto de UITI y de eventos por grupo. Sin vanos marcados quedan vacias a
proposito: un reparto de miles de vanos y otro de tres se dibujan igual, asi que caer al
circuito entero cambiaria el sujeto del panel en silencio.

## El HTML autocontenido, si se pide

Con `EXPORTAR_PANEL_WEB = True` la ultima celda ademas escribe
`reports/paneles/06_uiti_vano_explicabilidad_simulador.html` (~12 MB, con `plotly.js`
adentro) y, en local, lo abre en el navegador. En Databricks el archivo queda en el Volume
y no se abre nada, porque el driver no tiene navegador.

Ese HTML no es una foto: lleva el simulador funcionando. El forward del MIL corre en
JavaScript, transcrito de `chec_local_interpreter.mil_web_export.predecir_numpy`, que
`tests/test_mil_web_export.py` fija contra el modulo de torch; medido contra el modelo
real, u-hat coincide con 9e-6 de error relativo y ninguna clase cambia. Lo que si se acota
es la cobertura: los pesos son 0,46 MB y viajan siempre, pero la matriz de instancias son
88 MB para los 208 circuitos y no cabe en un archivo unico, asi que `CIRCUITOS_SIMULABLES`
elige cuales embarcar (por defecto, el circuito activo). Para un circuito que no viaje, el
panel lo dice en vez de fallar en silencio.

Los mapas del HTML traen ademas las mejoras del mapa de 01: un guion horizontal negro al
inicio y al fin de cada vano, y etiqueta de hover en cualquier punto del vano. Lo segundo
se calcula en el navegador: el hover de una traza de lineas en `Scattermap` se resuelve
contra los vertices, los tramos de `MVLINSEC` traen exactamente dos, y densificarlos del
lado del kernel pondria el peor circuito en 22.371 puntos y ~2,8 MB de etiquetas por capa,
por encima del limite que deja la figura en blanco.


## La matematica de lo que hace "Simular"

Todo lo de abajo describe UNA pulsacion del boton sobre la seleccion activa
$(c, w, M)$: circuito, ventana y conjunto de vanos marcados.

### 1. Las bolsas de la seleccion

La unidad de prediccion no es el evento: es la **bolsa**, la celda
$(\text{circuito}, \text{vano}, \text{ventana})$. Es la misma unidad en la que 04 define
la criticidad, y por eso el mapa simulado se puede comparar con el historico.

$$\mathcal{B}(c,w,M)=\{\,b=(c,v,w)\;:\;v\in M\,\},\qquad M=\varnothing\;\Rightarrow\;M:=V(c,w)$$

donde $V(c,w)$ son todos los vanos del circuito con al menos un evento en esa ventana:
sin vanos marcados el grano es el circuito completo, no un panel vacio.

Cada bolsa $b$ agrupa sus instancias $I_b$ -- las filas de evento de ese vano dentro de
esa ventana -- y trae dos cosas que **no se predicen nunca**:

$$n_b=|I_b|\quad(\text{eventos OBSERVADOS}),\qquad x_i\in\mathbb{R}^{p},\;p=80$$

Las $p=80$ columnas son 22 estructurales + 48 rezagos de clima + `COD_CAUSA` y sus 9
indicadores. Ojo con no confundir esa cuenta con la **particion por modalidades** que usa
el modelo, que no es la misma: `climaticos` son las 50 columnas de los 48 rezagos **mas
`DDT` y `NR_T`** (descargas y nivel de tormenta son clima, aunque viajen como columnas
estaticas), y `estructurales` son las 30 restantes -- las otras 20 estructurales mas
`COD_CAUSA` y sus 9 indicadores. El almacenamiento es CSR (`offsets`, `counts`), no una matriz rellenada:
el 52,7% de las bolsas son de un solo evento y el maximo es 46, asi que rellenar
desperdiciaria mas de 40x en la mitad de los datos. Al seleccionar, el indice de bolsa se
**renumera** desde 0, porque el modelo toma `n_bags = max(instance_bag)+1` y los ids
originales reservarian una bolsa vacia por cada celda no seleccionada.

**Los controles del simulador** actuan sobre las instancias, no sobre las bolsas. Un
control $\kappa$ gobierna un conjunto de columnas $F(\kappa)$ -- una sola para una
variable estructural, las 12 de una familia climatica -- y aplicarlo es

$$x_{i,j}\;\leftarrow\;\phi_j(\text{valor}),\qquad \forall\, i\in\textstyle\bigcup_b I_b,\;\forall\, j\in F(\kappa)$$

con $\phi_j$ la coercion a espacio de modelo (categoria por su codificador, fecha a
epoch, NaN a su centinela). No hay escalador despues: la matriz de instancias del MIL es
espacio crudo. **$n_b$ jamas se toca**: es un eje del espacio que define la clase, y
moverlo desplazaria al vano por una dimension que el modelo no predice.

### 2. La prediccion de clase de cada vano

El modelo hace **dos pasadas** sobre el mismo codificador. La primera existe solo para
producir las compuertas del grafo.

**(a) Codificacion y atencion.** Cada instancia se codifica por modalidad
(30 estructurales, 50 climaticas, segun la particion de arriba) y se concatena en
$z_i^{(1)}$. La bolsa se resume con
atencion tipo Ilse, normalizada dentro de la bolsa:

$$e_i=\mathbf{w}^{\top}\tanh(V z_i^{(1)}),\qquad
a_i=\frac{\exp(e_i)}{\sum_{i'\in I_b}\exp(e_{i'})},\qquad
z_b^{(1)}=\sum_{i\in I_b}a_i\,z_i^{(1)}$$

Esto es **invariante a la cardinalidad por construccion**: duplicar cada instancia de una
bolsa no cambia ningun $e_i$, el denominador se duplica, cada copia recibe $a_i/2$ y la
suma queda igual.

**(b) Compuertas del grafo experto.** Un decodificador lee el resumen de la bolsa y
produce una compuerta por arista:

$$g_b=2\,\sigma(W_g\,z_b^{(1)})\;\in\;(0,2)^{E},\qquad E=64$$

Se inicializa en cero, de modo que $g_b=\mathbf{1}$ al arrancar y el grafo aprendido
**parte exactamente del grafo experto fijo**.

**(c) Propagacion.** El grafo experto es una adyacencia fija $W\in\mathbb{R}^{80\times 80}$
con soporte en $E$ aristas. Cada instancia recibe, en la columna destino de cada arista:

$$x'_{i,j}\;=\;x_{i,j}\;+\;\alpha\!\!\sum_{e\,:\,\mathrm{dst}(e)=j}\!\! g_{b(i),e}\;w_e\;x_{i,\mathrm{src}(e)},
\qquad \alpha=0{,}2$$

Una columna que no es destino de ninguna arista queda intacta, exactamente.

**(d) Segunda pasada y fusion FiLM.** El MISMO codificador procesa $x'$, se vuelve a
agrupar con la MISMA atencion y las dos modalidades se fusionan modulando la
estructural con la climatica (`film_modulated_modality = estructurales`, del artefacto):

$$\hat z_b=z_b^{\text{est}}\odot\bigl(1+\gamma(z_b^{\text{clim}})\bigr)+\beta(z_b^{\text{clim}}),
\qquad p_b=h(\hat z_b),\qquad \hat u_b=\mathrm{expm1}(p_b)$$

La concatenacion es aditiva entre modalidades y no puede representar un producto entre una
variable estructural y una climatica; FiLM hace que el clima **reescale** lo estructural,
que es tambien la afirmacion de dominio: una rafaga pesa mas sobre un apoyo alto, viejo y
degradado. El modelo aprende en $\log(1+u)$ y se devuelve a UITI con `expm1`.

**(e) Clase.** Con el UITI predicho y los eventos observados, la clase sale de la
geometria KMeans de 04 -- **la misma que pinta el mapa base**, verificada al cargar el
modelo. En el espacio canonico `2` el eje de eventos es lineal y el de UITI logaritmico:

$$\zeta_b=\left(\frac{n_b-\mu_0}{s_0},\;\frac{\log_{10}\max(\hat u_b,\varepsilon)-\mu_1}{s_1}\right),
\qquad \hat k_b=\arg\min_{k\in\{0,1,2,3\}}\lVert \zeta_b-c_k\rVert^2$$

El mapa pinta $\hat k_b$ con la paleta de los cuatro grupos; lo que no tiene bolsa en la
ventana, o quedo fuera de la seleccion, va en negro. El simulador corre esto **dos veces**
-- sin y con los controles aplicados -- y $\Delta_b=\hat k_b^{\text{sim}}-\hat k_b^{\text{base}}$
es cuantos vanos cambian de clase.

### 3. El ranking de importancia de variables

Es un barrido min-max, **no SHAP**. Se mide sobre las mismas bolsas del mapa, con el
riesgo ordinal esperado: la distribucion suave sobre las cuatro clases, promediada.

$$P_{b,k}=\frac{\exp(-\lVert\zeta_b-c_k\rVert^2/\tau)}{\sum_{k'}\exp(-\lVert\zeta_b-c_{k'}\rVert^2/\tau)},
\qquad
R(X)=\frac{1}{|\mathcal{B}|}\sum_{b}\sum_{k}k\,P_{b,k}\;\in[0,3]$$

Para cada control **numerico** $\kappa$, con su rango observado $[m_\kappa,M_\kappa]$ en
todo el dataset, se lleva a sus dos extremos todas las columnas $F(\kappa)$ a la vez:

$$\Delta^{-}_{\kappa}=R\!\left(X^{\kappa\to m_\kappa}\right)-R(X),\qquad
\Delta^{+}_{\kappa}=R\!\left(X^{\kappa\to M_\kappa}\right)-R(X),\qquad
s_\kappa=\max\!\left(|\Delta^{-}_{\kappa}|,\,|\Delta^{+}_{\kappa}|\right)$$

Los controles categoricos y constantes **se omiten**: no tienen minimo/maximo numerico, e
inventarles un rango seria puntuar un escenario que nadie pidio (por eso el panel muestra
22 barras de 26 controles). Las barras son la normalizacion softmax, estable por resta del
maximo:

$$\rho_\kappa=\frac{\exp\bigl((s_\kappa-\max_{\kappa'}s_{\kappa'})/T\bigr)}{\sum_{\kappa'}\exp\bigl((s_{\kappa'}-\max s)/T\bigr)},\qquad T=1,\qquad \sum_\kappa\rho_\kappa=1$$

**Cuidado al leerlas**: el softmax aplana. Con este modelo el barrido mueve el riesgo
ordinal entre 0,02 y 0,08, asi que las 22 barras quedan cerca del 4,5% del reparto
uniforme. Eso dice "ninguna variable domina", y es una propiedad del modelo, no del panel.
La magnitud cruda $s_\kappa$ sigue en el hover.

### 4. El grafo inferido

Las compuertas de la parte (b) son lo unico del grafo que depende de la seleccion. Se
juntan en una matriz $G\in\mathbb{R}^{|\mathcal{B}|\times E}$, con $G_{b,e}=g_{b,e}$.

Antes de reconstruir nada se mide si esas compuertas **varian** entre vanos. Con
$\tilde G$ la matriz centrada por columnas y $\sigma_1,\dots$ sus valores singulares:

$$\mathrm{var}=\frac{1}{E}\sum_e \mathrm{Var}_b(G_{b,e}),\qquad
\mathrm{rank}_{\text{ef}}=\frac{\bigl(\sum_r\sigma_r^2\bigr)^2}{\sum_r\sigma_r^4},\qquad
\text{colapso}\iff \max_e \mathrm{std}_b(G_{b,e})<10^{-6}\;\lor\;\mathrm{rank}_{\text{ef}}\le 1$$

El rango efectivo es el cociente de participacion: vale $\approx 1$ cuando toda la
variacion vive en una sola direccion, es decir, cuando todos los vanos estan compuertados
igual. Un colapso **anula** el grafo y el panel lo dice, en vez de dibujar el grafo
experto fijo como si lo hubiera estimado esta seleccion. De ahi sale un limite duro:
con $|\mathcal{B}|<3$ la matriz centrada tiene rango 1 por construccion, asi que **menos
de 3 vanos nunca producen grafo**.

Si no hay colapso, el peso reconstruido de cada arista es el peso experto fijo tal como lo
usa esta familia de vanos:

$$\bar g_e=\frac{1}{|\mathcal{B}|}\sum_{b}G_{b,e},\qquad
A_{\mathrm{src}(e),\,\mathrm{dst}(e)}=\bar g_e\cdot w_e,\qquad A_{ij}=0 \text{ fuera del soporte}$$

El panel lo dibuja en disposicion circular sobre las variables que participan de al menos
una arista, con $A_{ij}$ en un marcador sobre el punto medio de cada arista.

### Presupuesto de una pulsacion

| Paso | Pasadas de bolsas |
|---|---|
| Mapa simulado (base + simulado) | 2 |
| Compuertas para el grafo | 1 |
| Importancia (base compartida + min/max por control) | $1+2K$, $K=22$ |
| **Total** | **48** |

Medido sobre el modelo real: 0,10 s para 24 bolsas, y 0,34 s solo de importancia para 537.
Por eso alcanza con un LRU de sesion y no hace falta cache en disco.